# anything needs to be downloaded

In [ ]:
import os, time, requests
from datetime import datetime, timedelta
import pandas as pd
import time
import re
import json
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import numpy as np
from pathlib import Path
import glob
from datetime import date
import re


In [ ]:
!pip install pandas plotly openpyxl ipywidgets kaleido

In [ ]:
pip install xlsxwriter

In [ ]:
!pip install pandas numpy TA-Lib tensorflow


In [ ]:
pip install numpy==1.23.5 --force-reinstall


In [ ]:
pip install --upgrade scipy numba


# Macro update code

In [ ]:
# ────────────────────────────────────────────────────────────────
# (실행 전 커널 재시작 권장)
import os
from datetime import datetime, timedelta
import pandas as pd
from pandas_datareader import data as pdr
import yfinance as yf

# ────────────────────────────────────────────────────────────────
# MACRO_DIR = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
MACRO_DIR = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
os.makedirs(MACRO_DIR, exist_ok=True)

INDICATORS = {
    "CPI":           ("fred",   "CPIAUCSL"),
    "WTI":           ("fred",   "MCOILWTICO"),
    "VIX":           ("fred",   "VIXCLS"),
    "FedFundsRate":      ("fred",   "FEDFUNDS"),
    "YieldCurve":    ("fred",  ("GS10","GS2")),
    "Unemployment":  ("fred",   "UNRATE"),
    "HY_Spread":     ("fred",   "BAMLH0A3HYCEY"),
}

def fetch_fred(series, start, end):
    return pdr.DataReader(series, "fred", start, end)

def fetch_yahoo(symbol, start, end):
    df = yf.download(
        symbol,
        start=start.strftime("%Y-%m-%d"),
        end  =end  .strftime("%Y-%m-%d"),
        progress=False
    )
    return df[["Close"]].rename(columns={"Close":"Value"})

for name, (source, code) in INDICATORS.items():
    print(f"\n▶ {name} 업데이트 시작")
    csv_old = None

    # 기존 파일 찾기
    for fn in os.listdir(MACRO_DIR):
        if fn.endswith(f" {name}.csv"):
            csv_old = os.path.join(MACRO_DIR, fn)
            break

    # 1) 첫 생성인지
    if csv_old is None:
        print("  – 초기 파일 생성")
        start = datetime(1970,1,1)
    else:
        df_old = pd.read_csv(csv_old, parse_dates=["Date"])
        start   = df_old["Date"].max() + timedelta(days=1)

    end = datetime.today() + timedelta(days=1)

    # 2) 새로운 데이터 가져오기 (실패해도 df는 빈 DF)
    try:
        if source == "fred":
            if isinstance(code, tuple):
                df1 = fetch_fred(code[0], start, end)
                df2 = fetch_fred(code[1], start, end)
                df = pd.DataFrame({
                    "Date": df1.index,
                    "Value": df1[code[0]] - df2[code[1]]
                })
            else:
                df = fetch_fred(code, start, end).reset_index()
                df.columns = ["Date","Value"]
        else:  # yahoo
            df = fetch_yahoo(code, start, end).reset_index()
            df.columns = ["Date","Value"]
    except Exception as e:
        print(f"  ✖ {name} 데이터 fetch 실패: {e}")
        df = pd.DataFrame(columns=["Date","Value"])

    if df.empty:
        print("  – 신규 데이터 없음, 기존 데이터만으로 정렬·저장")

    # 3) 합치고 중복 제거 후 내림차순 정렬
    if csv_old is None:
        combined = df.copy()
    else:
        combined = pd.concat([df_old, df], ignore_index=True)

    combined = (
        combined
        .drop_duplicates(subset="Date", keep="first")
        .sort_values("Date", ascending=False)
        .reset_index(drop=True)
    )

    # CPI만 YoY 계산
    if name == "CPI":
        asc = combined.sort_values("Date").reset_index(drop=True)
        asc["YoY_%"] = (asc["Value"].pct_change(12) * 100).round(2)
        combined = asc.sort_values("Date", ascending=False).reset_index(drop=True)

    # 4) 파일명에 기간 붙이고 저장
    start_str = combined["Date"].min().strftime("%Y.%m.%d")
    end_str   = combined["Date"].max().strftime("%Y.%m.%d")
    new_fn    = f"{start_str}_{end_str} {name}.csv"
    new_path  = os.path.join(MACRO_DIR, new_fn)

    combined.to_csv(new_path, index=False, date_format="%Y-%m-%d")
    print(f"  ✔ 저장: {new_fn}")

    # 5) 이전 파일 삭제
    if csv_old and new_path != csv_old:
        os.remove(csv_old)


# 너가 원하는 티커를 적어 아래에

In [ ]:
def get_ticker_map():
    """
    return dict: { Ticker: (company_slug, fiscal_year_end_month) }
    """
    return {
        "GOOG": ("alphabet", 12),
        "BN": ("brookfield", 12),
        "OXY": ("occidental-petroleum", 12),
        "CRSP": ("crispr-therapeutics-ag", 12),
        "NVO": ("novo-nordisk", 12),
        "PATH": ("uipath", 12),
        "OKTA": ("okta", 12),
        "ILMN": ("illumina", 12),
        "TSM": ("taiwan-semiconductor-manufacturing", 12),
        "CRCL": ("circle-internet", 12),
        "NTRA": ("natera", 12),
        "META": ("meta-platforms", 12),
        "NU": ("nu-holdings", 12),
        "CPNG": ("coupang", 12),
        "CVX": ("chevron", 12),
        "PLTR": ("palantir-technologies", 12),
        "DDOG": ("datadog", 12),
        "CRWD": ("crowdstrike", 12),
        "AVGO": ("broadcom", 12),
        "UNH": ("unitedhealth-group", 12),
        "LMND": ("lemonade", 12),
        "AAPL": ("apple", 9),   # 🔹 애플: 회계연도 9월 종료
        "MITK": ("mitek-systems", 12),
        "MSFT": ("microsoft", 6),  # 🔹 마이크로소프트: 6월 종료
        "BBAI": ("bigbearai-holdings", 12),
        "VST": ("vistra", 12),
        "VRT": ("vertiv-holdings", 12),
        "MP": ("mp-materials", 12),
        "COST": ("costco", 8),  # 🔹 코스트코: 8월 종료
        "TEVA": ("teva-pharmaceutical-industries", 12),
        "TSLA": ("tesla", 12),
        "U": ("unity-software", 12),
        "RDDT": ("reddit", 12),
        "SNOW": ("snowflake", 1),  # 🔹 스노우플레이크: 1월 종료
        "RKLB": ("rocket-lab", 12),
        "MELI": ("mercadolibre", 12),
        "EH": ("ehang-holdings", 12),
        "GRAL": ("grail", 12),
        "PLUG": ("plug-power", 12),
        "SE": ("sea", 12),
        "TEM": ("tempus-ai", 12),
        "IREN": ("iren", 12),
        "PI": ("impinj", 12),
        "JOBY": ("joby-aviation", 12),
        "APP": ("applovin", 12),
        "COIN": ("coinbase-global", 12),
        "BTC-USD": ("bitcoin", 12),   # 크립토는 보통 12월로 placeholder
        "XRP-USD": ("xrp", 12),
        "NVDA": ("nvidia", 1),  # 🔹 엔비디아: 1월 종료
    }


# Stock Price History Update

In [ ]:
ROOT_DIR = r"C:\Users\LabPC\OneDrive\주식\Back Test"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

def to_epoch(dt: datetime) -> int:
    return int(time.mktime(dt.timetuple()))

def format_vol(x):
    if pd.isna(x): return ""
    v = int(x)
    if v >= 1_000_000: return f"{v/1_000_000:.2f}M"
    if v >=   1_000:   return f"{v/1_000:.2f}K"
    return str(v)

def fetch_chart_json(ticker, start_dt, end_dt):
    p1, p2 = to_epoch(start_dt), to_epoch(end_dt)
    url = (
        f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
        f"?period1={p1}&period2={p2}&interval=1d&includePrePost=false"
    )
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    data = r.json()
    if "chart" not in data or not data["chart"]["result"]:
        return pd.DataFrame()
    res = data["chart"]["result"][0]
    ts = res.get("timestamp", [])
    q  = res.get("indicators", {}).get("quote", [{}])[0]
    if not ts or "close" not in q:
        return pd.DataFrame()
    return pd.DataFrame({
        "Date":  [datetime.fromtimestamp(t) for t in ts],
        "Open":  q.get("open"),
        "High":  q.get("high"),
        "Low":   q.get("low"),
        "Price": q.get("close"),
        "Volume":q.get("volume"),
    })

# ───────────────────────────────────────────────────────────
# 1) 폴더 생성 및 초기 CSV (전체 히스토리)  — 튜플 언패킹 (slug, fye)
# ───────────────────────────────────────────────────────────
for ticker, (company_slug, fye) in get_ticker_map().items():
    name = company_slug.replace("-", " ").title()
    folder = os.path.join(ROOT_DIR, name)
    os.makedirs(folder, exist_ok=True)

    base_fn = f"{name} Historical Data.csv"
    base_path = os.path.join(folder, base_fn)

    if not os.path.exists(base_path):
        print(f"▶ {name}: 초기 CSV 생성 중…")
        df_all = fetch_chart_json(ticker, datetime(1970,1,1), datetime.today()+timedelta(days=1))
        if df_all.empty:
            print(f"  ✖ {ticker} 데이터 없음")
            continue
        df_all["Vol."]     = df_all["Volume"].apply(format_vol)
        df_all["Change %"] = (df_all["Price"].pct_change()*100).map(lambda x: f"{x:.2f}%")
        df_all = df_all[["Date","Price","Open","High","Low","Vol.","Change %"]]
        df_all["Date"] = df_all["Date"].dt.strftime("%m/%d/%Y")
        df_all.to_csv(base_path, index=False, encoding="utf-8-sig")
        print(f"  ✔ {base_fn} 생성 완료")

# ───────────────────────────────────────────────────────────
# 2) 기존 CSV 업데이트 및 파일명 재생성 — 튜플 언패킹 (slug, fye)
# ───────────────────────────────────────────────────────────
for ticker, (company_slug, fye) in get_ticker_map().items():
    name = company_slug.replace("-", " ").title()
    folder = os.path.join(ROOT_DIR, name)

    base_fn = f"{name} Historical Data.csv"
    base_path = os.path.join(folder, base_fn)
    if not os.path.exists(base_path):
        continue

    print(f"\n▶ {name} ({ticker}) 업데이트 중…")
    df_old = pd.read_csv(base_path, parse_dates=["Date"])
    last = df_old["Date"].max()
    start = last + timedelta(days=1)
    end = datetime.today() + timedelta(days=1)

    df_new = pd.DataFrame()
    if start.date() < end.date():
        tmp = fetch_chart_json(ticker, start, end)
        if not tmp.empty:
            tmp["Vol."]     = tmp["Volume"].apply(format_vol)
            tmp["Change %"] = (tmp["Price"].pct_change()*100).map(lambda x: f"{x:.2f}%")
            df_new = tmp[["Date","Price","Open","High","Low","Vol.","Change %"]]
            print(f"  ✔ {len(df_new)}개 신규 행 추가")
        else:
            print("  – 신규 데이터 없음")
    else:
        print("  – 신규 데이터 없음")

    # 병합 및 정리
    df_combined = pd.concat([df_old, df_new], ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset="Date").sort_values("Date", ascending=False)

    # 날짜 범위와 포맷 설정
    max_d = df_combined["Date"].max().strftime("%Y.%m.%d")
    min_d = df_combined["Date"].min().strftime("%Y.%m.%d")
    df_combined["Date"] = df_combined["Date"].dt.strftime("%m/%d/%Y")

    # 새 파일명 생성 및 저장
    new_fn = f"{max_d}-{min_d} {name} Historical Data.csv"
    new_path = os.path.join(folder, new_fn)
    df_combined.to_csv(new_path, index=False, encoding="utf-8-sig")

    # 이전 CSV 파일 제거, 새 파일만 남김
    for f in os.listdir(folder):
        if f.endswith(".csv") and f != new_fn:
            os.remove(os.path.join(folder, f))

    print(f"  ✔ 저장 완료: {new_fn}")

# price Data Processing

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# 0) 경로
# ─────────────────────────────────────────────────────────────
RAW_ROOT = r"C:\Users\seung\OneDrive\주식\Back Test"        # 종목별 폴더가 있는 루트
PROCESSED_FOLDER = r"C:\Users\seung\OneDrive\주식\Processed Data"
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 1) 유틸: 볼륨/퍼센트 파서 & 컬럼 표준화
# ─────────────────────────────────────────────────────────────
def _parse_volume(v):
    """'3.36M', '8.26M', '123.4K', '1.2B' -> 정수 주식 수"""
    if pd.isna(v): return np.nan
    s = str(v).strip().replace(',', '')
    mult = 1
    if s.endswith(('K','k')): mult, s = 1_000, s[:-1]
    elif s.endswith(('M','m')): mult, s = 1_000_000, s[:-1]
    elif s.endswith(('B','b')): mult, s = 1_000_000_000, s[:-1]
    try:
        return float(s) * mult
    except:
        return np.nan

def _parse_change(p):
    """'0.62%' -> 0.62 (percent 단위 실수)"""
    if pd.isna(p): return np.nan
    s = str(p).strip().replace('%','')
    try:
        return float(s)
    except:
        return np.nan

def normalize_columns(df):
    """
    원본 열: Date, Price, Open, High, Low, Vol., Change %
    표준화 열: 날짜, 종가, 시가, 고가, 저가, 거래량, 등락률(%)
    """
    # 느슨한 매핑(공백/대소문자/마침표 대비)
    rename_map = {}
    for c in df.columns:
        c_clean = c.strip().lower().replace(' ','').replace('.','')
        if c_clean == 'date':         rename_map[c] = '날짜'
        elif c_clean == 'price' or c_clean == 'close': rename_map[c] = '종가'
        elif c_clean == 'open':       rename_map[c] = '시가'
        elif c_clean == 'high':       rename_map[c] = '고가'
        elif c_clean == 'low':        rename_map[c] = '저가'
        elif c_clean in ('vol', 'volume'): rename_map[c] = '거래량(raw)'
        elif c_clean in ('change%','changepct','change'): rename_map[c] = '등락률(%)'
    df = df.rename(columns=rename_map)

    # 필수 컬럼만 남기고 타입 정리
    if '날짜' in df.columns:
        df['날짜'] = pd.to_datetime(df['날짜'], errors='coerce')
    if '종가' in df.columns:
        df['종가'] = pd.to_numeric(df['종가'], errors='coerce')
    if '시가' in df.columns:
        df['시가'] = pd.to_numeric(df['시가'], errors='coerce')
    if '고가' in df.columns:
        df['고가'] = pd.to_numeric(df['고가'], errors='coerce')
    if '저가' in df.columns:
        df['저가'] = pd.to_numeric(df['저가'], errors='coerce')
    if '거래량(raw)' in df.columns:
        df['거래량'] = df['거래량(raw)'].map(_parse_volume)
        df.drop(columns=['거래량(raw)'], inplace=True)
    if '등락률(%)' in df.columns:
        df['등락률(%)'] = df['등락률(%)'].map(_parse_change)

    # 핵심만 반환
    keep = [c for c in ['날짜','종가','시가','고가','저가','거래량','등락률(%)'] if c in df.columns]
    df = df[keep].dropna(subset=['날짜','종가']).sort_values('날짜').reset_index(drop=True)
    return df

# ─────────────────────────────────────────────────────────────
# 2) 지표 계산
# ─────────────────────────────────────────────────────────────
def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    req = {'종가', '거래량'}
    if not req.issubset(df.columns):
        missing = req - set(df.columns)
        raise ValueError(f"지표 계산 필수 컬럼 누락: {missing}")

    out = df.copy()

    # Bollinger Bands (20일)
    out['BB_MID']   = out['종가'].rolling(20).mean()
    out['BB_STD']   = out['종가'].rolling(20).std(ddof=0)
    out['BB_UPPER'] = out['BB_MID'] + 2 * out['BB_STD']
    out['BB_LOWER'] = out['BB_MID'] - 2 * out['BB_STD']

    # MACD (12,26,9)
    ema12 = out['종가'].ewm(span=12, adjust=False).mean()
    ema26 = out['종가'].ewm(span=26, adjust=False).mean()
    out['MACD']     = ema12 - ema26
    out['MACD_SIG'] = out['MACD'].ewm(span=9, adjust=False).mean()

    # RSI (14) & Signal (9) — 단순 이동평균 기반
    delta = out['종가'].diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_g = gain.rolling(14).mean()
    avg_l = loss.rolling(14).mean()
    rs    = avg_g / avg_l
    out['RSI_14']   = 100 - (100 / (1 + rs))
    out['RSI_SIG9'] = out['RSI_14'].rolling(9).mean()

    # OBV & Signal (9)
    out['OBV']      = (np.sign(out['종가'].diff()) * out['거래량']).fillna(0).cumsum()
    out['OBV_SIG9'] = out['OBV'].rolling(9).mean()

    return out.dropna()

# ─────────────────────────────────────────────────────────────
# 3) 종목 폴더별 전체 CSV 취합 → 표준화 → 지표 계산 → 저장
#    예) ...\Back Test\Meta Platforms\*.csv  →  ...\Processed Data\Meta Platforms_지표포함.csv
# ─────────────────────────────────────────────────────────────
def load_csv_with_fallback(path):
    """utf-8-sig 우선, 안되면 cp949로 재시도"""
    for enc in ('utf-8-sig','cp949'):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    # 마지막 시도: 인코딩 자동 추정 없이 기본으로
    return pd.read_csv(path)

if __name__ == "__main__":
    # 종목 폴더 탐색
    tickers = [d for d in os.listdir(RAW_ROOT) if os.path.isdir(os.path.join(RAW_ROOT, d))]
    print("🔎 발견된 종목 폴더:", tickers)

    for comp in tickers:
        comp_dir = os.path.join(RAW_ROOT, comp)
        csvs = glob.glob(os.path.join(comp_dir, "*.csv"))
        if not csvs:
            print(f"⚠️ {comp}: CSV 없음, 스킵")
            continue

        # 여러 파일이 있을 수 있으니 전부 로드→정규화→병합
        frames = []
        for p in csvs:
            try:
                df_raw = load_csv_with_fallback(p)
                df_norm = normalize_columns(df_raw)
                frames.append(df_norm)
                print(f"📥 {comp} ← {os.path.basename(p)}  rows={len(df_norm)}")
            except Exception as e:
                print(f"❌ {comp}: {os.path.basename(p)} 로드 실패 → {e}")

        if not frames:
            print(f"⚠️ {comp}: 유효 데이터 없음, 스킵")
            continue

        base = pd.concat(frames, ignore_index=True)
        # 날짜 중복 제거(가장 최신 값 우선)
        base = base.sort_values('날짜').drop_duplicates(subset=['날짜'], keep='last').reset_index(drop=True)

        # 거래량이 비어있으면 지표 계산 불가 → 경고
        if '거래량' not in base.columns or base['거래량'].isna().all():
            print(f"⚠️ {comp}: 거래량 데이터가 없어 지표 계산 불가. 표준화만 저장합니다.")
            save_df = base
        else:
            save_df = compute_indicators(base)

        save_path = os.path.join(PROCESSED_FOLDER, f"{comp}_지표포함.csv")
        save_df.to_csv(save_path, index=False, encoding="utf-8-sig")
        print(f"✅ 저장 완료: {save_path} (rows={len(save_df)})")


# Finacial Data Crwaling Code from Macrotrend

In [ ]:
freq = "Q"
save_folder = r"C:\Users\LabPC\OneDrive\주식\Financial_Data_real"
os.makedirs(save_folder, exist_ok=True)

# Selenium 설정
opts = Options()
opts.add_argument("--window-size=1920,1200")
# opts.add_argument("--headless")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

statements = {
    "Income Statement": "income-statement",
    "Balance Sheet": "balance-sheet",
    "Cash Flow Statement": "cash-flow-statement",
    "Key Financial Ratios": "financial-ratios"
}

def fetch_statement_df(ticker, company_slug, statement, freq="Q"):
    url = f"https://www.macrotrends.net/stocks/charts/{ticker}/{company_slug}/{statement}?freq={freq}"
    driver.get(url)
    time.sleep(5)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    script_tags = soup.find_all("script", string=re.compile(r"originalData"))
    target = None
    for sc in script_tags:
        m = re.search(r"var originalData = (\[.*?\]);", sc.string or "", re.S)
        if m:
            target = m.group(1)
            break
    if not target:
        raise ValueError(f"originalData not found for {ticker} - {statement}")
    data = json.loads(target)
    rows = []
    for e in data:
        row = {"Metric": BeautifulSoup(e.get("field_name", ""), "html.parser").get_text()}
        for k, v in e.items():
            if re.match(r"\d{4}-\d{2}-\d{2}", k):
                row[k] = v
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Metric")
    return df.sort_index(axis=1, ascending=False)

# ▶ 모든 티커 순회 (튜플 언패킹)
ticker_map = get_ticker_map()

for ticker, (company_slug, fye) in ticker_map.items():
    print(f"\n🔄 Processing {ticker} ({company_slug}), FYE={fye}...")

    dfs = {}
    for sheet_name, path in statements.items():
        print(f"   📊 Fetching: {sheet_name}...")
        try:
            df = fetch_statement_df(ticker, company_slug, path, freq)
            dfs[sheet_name] = df
        except Exception as e:
            print(f"   ❌ Error fetching {sheet_name} for {ticker}: {e}")

    # 저장
    if dfs:
        output_file = os.path.join(save_folder, f"{ticker}_financials_{freq}.xlsx")
        with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
            for sheet, df in dfs.items():
                df.to_excel(writer, sheet_name=sheet)
        print(f"   ✅ 저장 완료: {output_file}")
    else:
        print(f"   ⚠️ No data for {ticker}")

driver.quit()

# Financial Summary

In [ ]:


# # ====== DCF 공통 가정 ======
# WACC = 0.10
# COE = 0.10
# G_SHORT = 0.05
# PROJ_YEARS = 5
# G_TERM = 0.025

# # ====== 경로 ======
# BASE = Path(r"C:\Users\seung\OneDrive\주식")
# BACKTEST_DIR = BASE / "Back Test"
# FIN_DIR = BASE / "Financial_Data_real"
# SUMMARY_DIR = FIN_DIR / "Summary"
# SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
# PROCESSED_DIR = Path(r"C:\Users\seung\OneDrive\주식\Processed Data")

# # ====== 유틸 ======
# def canonical_company_display(name_or_slug: str) -> str:
#     """'crispr-therapeutics-ag' -> 'Crispr Therapeutics Ag'"""
#     disp = name_or_slug.replace("-", " ").strip()
#     disp = re.sub(r"\s+", " ", disp)
#     return disp.title()

# def find_processed_csv(display_name: str):
#     # 예: "Alphabet_지표포함.csv", "Alphabet_지표포함_2025-08-28.csv" 등 유연 매칭
#     patterns = [
#         str(PROCESSED_DIR / f"{display_name}_지표포함*.csv"),
#         str(PROCESSED_DIR / f"{display_name}*_지표포함*.csv"),
#     ]
#     for pat in patterns:
#         hits = glob.glob(pat)
#         if hits:
#             # 가장 최근 수정 파일 선택
#             hits.sort(key=os.path.getmtime, reverse=True)
#             return hits[0]
#     return None

# def safe_div(n: pd.Series, d: pd.Series, eps: float = 1e-9) -> pd.Series:
#     n = pd.to_numeric(n, errors="coerce")
#     d = pd.to_numeric(d, errors="coerce")
#     d_safe = d.where(d.abs() > eps)
#     return n / d_safe

# def dcf_equity_value_from_fcf(fcf0, net_debt, rate, g_short, years, g_term):
#     if pd.isna(fcf0) or fcf0 <= 0:
#         return np.nan
#     pv = sum(fcf0 * (1 + g_short)**t / (1 + rate)**t for t in range(1, years+1))
#     cf_N = fcf0 * (1 + g_short)**years
#     tv = cf_N * (1 + g_term) / (rate - g_term)
#     pv_tv = tv / (1 + rate)**years
#     return pv + pv_tv - (net_debt if pd.notna(net_debt) else 0)

# def pct_change_abs(series, periods=1):
#     prev = series.shift(periods)
#     return (series - prev) / prev.abs() * 100

# def pick_series(df: pd.DataFrame, candidates):
#     """후보 열 중 존재하는 첫 숫자 Series 반환 (없으면 None)"""
#     for c in candidates:
#         if c in df.columns:
#             s = pd.to_numeric(df[c], errors="coerce")
#             if s.notna().any():
#                 return s
#     return None

# def get_num(row: pd.Series, col: str):
#     """row[col]을 float로, 없거나 비수치면 NaN"""
#     if col in row.index and pd.notna(row[col]):
#         try:
#             return float(row[col])
#         except Exception:
#             return np.nan
#     return np.nan


# # ====== 메인 분석 ======
# def analyze_company(TICKER, COMPANY_SLUG, FY_END_MONTH, MARKET_PRICE_OVERRIDE=None):
#     print(f"\n▶ 분석 시작: {TICKER} ({COMPANY_SLUG})")

#     # --- 폴더/파일명 정규화 (슬러그 → Title Case) ---
#     display_name = canonical_company_display(COMPANY_SLUG)  # 'Crispr Therapeutics Ag'
#     company_folder = BACKTEST_DIR / display_name
#     price_pattern = str(company_folder / f"* {display_name} Historical Data.csv")

#     price_candidates = glob.glob(price_pattern)
#     if not price_candidates and company_folder.exists():
#         # 폴백: 폴더 내 '* Historical Data.csv' 중 display_name 포함
#         price_candidates = [
#             str(p) for p in company_folder.glob("* Historical Data.csv")
#             if display_name.lower() in p.name.lower()
#         ]
#     if not price_candidates:
#         print(f"❌ 가격 CSV 없음: {display_name}")
#         return
#     PRICE_CSV = price_candidates[0]

#     FIN_XLSX = FIN_DIR / f"{TICKER}_financials_Q.xlsx"
#     if not FIN_XLSX.exists():
#         print(f"❌ 재무 엑셀 없음: {TICKER}")
#         return

#     # 재무 시트 로드
#     def load_financials(fp):
#         df = lambda sh: (pd.read_excel(fp, sheet_name=sh, index_col=0)
#                            .T.reset_index().rename(columns={"index": "Date"})
#                            .assign(Date=lambda x: pd.to_datetime(x["Date"])))
#         return df("Income Statement"), df("Balance Sheet"), df("Cash Flow Statement"), df("Key Financial Ratios")

#     inc, bal, cf, rat = load_financials(FIN_XLSX)
#     q = inc.merge(bal, on="Date", how="left").merge(cf, on="Date", how="left").merge(rat, on="Date", how="left")
#     q.sort_values("Date", inplace=True)

#     # 가격 매칭 + Full Price 보존
#     price_full = pd.read_csv(PRICE_CSV)
#     price_full["Date"] = pd.to_datetime(price_full["Date"])
#     price_full.rename(columns=lambda c: c.strip(), inplace=True)
#     price_full.sort_values("Date", inplace=True)
#     price_col = next(c for c in price_full.columns if "Price" in c)
#     px = price_full[["Date", price_col]].rename(columns={price_col: "Price"})
#     q = pd.merge_asof(q, px, on="Date", direction="backward")
#     if MARKET_PRICE_OVERRIDE is not None:
#         q["Price"] = float(MARKET_PRICE_OVERRIDE)

#     # -------- 분기별 계산 --------
#     # EPS / Shares / Dividend per share 시리즈 구성 (빈칸 방지)
#     eps_series = pick_series(q, ["EPS - Earnings Per Share", "Basic EPS"])
#     shares_series = pick_series(q, ["Shares Outstanding", "Basic Shares Outstanding"])

#     # EPS가 없으면 Net Income / Shares로 대체
#     if eps_series is None:
#         if ("Net Income" in q.columns) and (shares_series is not None):
#             eps_series = pd.to_numeric(q["Net Income"], errors="coerce") / shares_series
#         else:
#             eps_series = pd.Series(np.nan, index=q.index)

#     price_series = pd.to_numeric(q["Price"], errors="coerce")
#     div_total_q = pd.to_numeric(q.get("Common Stock Dividends Paid", np.nan), errors="coerce").abs()
#     div_ps_q = safe_div(div_total_q, shares_series)
#     div_yield_q = safe_div(div_ps_q, price_series) * 100

#     liab = pd.to_numeric(q.get("Total Liabilities", np.nan), errors="coerce")
#     cash = pd.to_numeric(q.get("Cash On Hand", np.nan), errors="coerce")
#     ev_q = (price_series * shares_series) + liab - cash
#     ebit_q = pd.to_numeric(q.get("EBIT", np.nan), errors="coerce")
#     ev_ebit_q = safe_div(ev_q, ebit_q)

#     def calc_q(r):
#         pr  = get_num(r, "Price")
#         sh  = get_num(r, "Shares Outstanding")
#         if np.isnan(sh):
#             sh = get_num(r, "Basic Shares Outstanding")
#         tca = get_num(r, "Total Current Assets")
#         tl  = get_num(r, "Total Liabilities")
#         she = get_num(r, "Share Holder Equity")
#         ni  = get_num(r, "Net Income")
#         ebit= get_num(r, "EBIT")
#         dep = get_num(r, "Total Depreciation And Amortization - Cash Flow")
#         capx= get_num(r, "Net Change In Property, Plant, And Equipment")
#         dwc = get_num(r, "Total Change In Assets/Liabilities")
#         div = abs(get_num(r, "Common Stock Dividends Paid"))

#         ncav = (tca - tl) if (pd.notna(tca) and pd.notna(tl)) else np.nan
#         eps  = (ni / sh) if (pd.notna(ni) and pd.notna(sh) and sh != 0) else np.nan
#         graham = np.sqrt(22.5 * max(eps, 0) * max((she / sh) if (pd.notna(she) and pd.notna(sh) and sh != 0) else 0, 0)) \
#                  if pd.notna(sh) else np.nan

#         fcff = (ebit*0.75 + dep - (-capx) - (-dwc)) if pd.notna(ebit) else np.nan
#         owner = (ni + dep + capx + dwc) if pd.notna(ni) else np.nan
#         ddm = (div * (1 + G_SHORT) / (COE - G_SHORT)) if pd.notna(div) and div > 0 else np.nan

#         mkcap = pr * sh if (pd.notna(pr) and pd.notna(sh)) else np.nan
#         underv_ncav = (mkcap < ncav) if (pd.notna(mkcap) and pd.notna(ncav)) else np.nan

#         return pd.Series({
#             "Price": pr,
#             "Market Cap": mkcap,
#             "NCAV": ncav,
#             "Undervalued by NCAV": underv_ncav,
#             "Graham Number": graham,
#             "FCFF": fcff,
#             "Owner Earnings": owner,
#             "DDM Value": ddm
#         })

#     # 기본 분기 테이블
#     quarterly = pd.concat([q[["Date"]], q.apply(calc_q, axis=1)], axis=1).set_index("Date")

#     # === QoQ 핵심 지표 (빈칸 방지) ===
#     quarterly["EPS_Q"] = eps_series.values
#     eps_q_growth_qoq = (eps_series - eps_series.shift(1)) / eps_series.shift(1).abs() * 100
#     pe_q = safe_div(price_series, eps_series)
#     peg_q = safe_div(pe_q, eps_q_growth_qoq)
#     pegy_q = safe_div(pe_q, eps_q_growth_qoq + div_yield_q)

#     quarterly["EPS_Growth_QoQ_%"] = eps_q_growth_qoq.values
#     quarterly["PE_Q"] = pe_q.values
#     quarterly["Div_Yield_Q_%"] = div_yield_q.values
#     quarterly["PEG_Q"] = peg_q.values
#     quarterly["PEGY_Q"] = pegy_q.values
#     quarterly["EV/EBIT_Q"] = ev_ebit_q.values

#     # 보기 좋은 컬럼 순서
#     cols_order = [
#         "Price", "Market Cap", "NCAV", "Undervalued by NCAV",
#         "EPS_Q", "EPS_Growth_QoQ_%", "PE_Q", "Div_Yield_Q_%", "PEG_Q", "PEGY_Q", "EV/EBIT_Q",
#         "Graham Number", "FCFF", "Owner Earnings", "DDM Value"
#     ]
#     quarterly = quarterly[[c for c in cols_order if c in quarterly.columns]]


#     # -------- TTM 계산 --------
#     qq = q.set_index("Date").sort_index()
#     ttm = lambda col: qq[col].rolling(4, min_periods=4).sum()
#     net = ttm("Net Income"); ebit = ttm("EBIT"); dep = ttm("Total Depreciation And Amortization - Cash Flow")
#     capex = (-qq["Net Change In Property, Plant, And Equipment"]).rolling(4).sum()
#     delta_wc = (-qq["Total Change In Assets/Liabilities"]).rolling(4).sum()
#     div_ps = (abs(qq["Common Stock Dividends Paid"]) / qq["Shares Outstanding"]).rolling(4).sum()
#     snap = qq[["Total Liabilities", "Cash On Hand", "Shares Outstanding",
#                "Total Current Assets", "Total Assets", "Price"]]

#     ncav_ttm = snap["Total Current Assets"] - snap["Total Liabilities"]
#     market_cap_ttm = snap["Price"] * snap["Shares Outstanding"]
#     roic = ebit / (snap["Total Current Assets"] + (snap["Total Assets"] - snap["Total Current Assets"]))
#     ev = market_cap_ttm + snap["Total Liabilities"] - snap["Cash On Hand"]
#     ev_ebit = ev / ebit

#     eps_ttm = net / snap["Shares Outstanding"]
#     pe_ttm = snap["Price"] / eps_ttm
#     eps_yoy = pct_change_abs(eps_ttm, periods=4)
#     div_yield_ttm = div_ps / snap["Price"] * 100
#     peg_ttm = safe_div(pe_ttm, eps_yoy)
#     pegy_ttm = safe_div(pe_ttm, eps_yoy + div_yield_ttm)

#     # DCF
#     net_debt = snap["Total Liabilities"] - snap["Cash On Hand"]
#     dcf_fcff = pd.Series({
#         dt: dcf_equity_value_from_fcf(
#             (ebit.loc[dt] * 0.75 + dep.loc[dt] - capex.loc[dt] - delta_wc.loc[dt]),
#             net_debt.loc[dt], WACC, G_SHORT, PROJ_YEARS, G_TERM
#         ) for dt in snap.index
#     })
#     dcf_owner = pd.Series({
#         dt: dcf_equity_value_from_fcf(
#             (net.loc[dt] + dep.loc[dt] - capex.loc[dt] - delta_wc.loc[dt]),
#             0.0, COE, G_SHORT, PROJ_YEARS, G_TERM
#         ) for dt in snap.index
#     })
#     dcf_fcff_price = dcf_fcff / snap["Shares Outstanding"]
#     dcf_owner_price = dcf_owner / snap["Shares Outstanding"]

#     yearly = pd.DataFrame({
#         "Market Cap": market_cap_ttm,
#         "NCAV_TTM": ncav_ttm,
#         "Undervalued by NCAV": market_cap_ttm < ncav_ttm,
#         "ROIC": roic,
#         "EV/EBIT": ev_ebit,
#         "EPS_TTM": eps_ttm, "PE_TTM": pe_ttm, "EPS_Growth_YoY_%": eps_yoy,
#         "PEG_TTM": peg_ttm, "PEGY_TTM": pegy_ttm,
#         "Graham Number": quarterly["Graham Number"],
#         "DDM Value": quarterly["DDM Value"],
#         "DCF_FCFF_Price": dcf_fcff_price,
#         "DCF_OwnerEarnings_Price": dcf_owner_price,
#         "Altman Z": (1.2 * ((snap["Total Current Assets"] - snap["Total Liabilities"]) / snap["Total Assets"]) +
#                      1.4 * ((qq["Retained Earnings (Accumulated Deficit)"]) / snap["Total Assets"]) +
#                      3.3 * (ebit / snap["Total Assets"]) +
#                      0.6 * (market_cap_ttm / snap["Total Liabilities"]) +
#                      1.0 * (net / snap["Total Assets"]))
#     })

#     # 연말 스냅샷 (모든 월 지원)
#     fq_map = {
#         1:'Q-JAN', 2:'Q-FEB', 3:'Q-MAR', 4:'Q-APR', 5:'Q-MAY', 6:'Q-JUN',
#         7:'Q-JUL', 8:'Q-AUG', 9:'Q-SEP', 10:'Q-OCT', 11:'Q-NOV', 12:'Q-DEC'
#     }
#     fq = yearly.index.to_period(fq_map.get(FY_END_MONTH, 'Q-DEC')).quarter
#     snapshot = yearly[fq == 4]

#     # 저장 (Quarterly + Yearly_TTM + Inputs + Price_History)
#     processed_path = find_processed_csv(display_name)
#     if processed_path:
#         price_hist_df = pd.read_csv(processed_path, encoding="utf-8-sig")
#         # '날짜'가 있으면 정렬만(열 이름은 그대로 보존)
#         if '날짜' in price_hist_df.columns:
#             price_hist_df['날짜'] = pd.to_datetime(price_hist_df['날짜'], errors='coerce')
#             price_hist_df = price_hist_df.sort_values('날짜')
#     else:
#         print(f"⚠ 가공본 없음: {display_name} → Price_History는 원본 CSV로 대체")
#         price_hist_df = price_full  # 폴백

#     with pd.ExcelWriter(out, engine="xlsxwriter") as w:
#         quarterly.round(4).to_excel(w, sheet_name="Quarterly")
#         snapshot.round(4).to_excel(w, sheet_name="Yearly_TTM")
#         q.to_excel(w, sheet_name="Inputs", index=False)
#         # 여기만 변경: 지표포함 CSV를 Price_History로 저장(열 이름/구조 보존)
#         price_hist_df.to_excel(w, sheet_name="Price_History", index=False)
#     print(f"✅ 저장됨: {out}")

# # ========== 전체 실행 ==========
# for t, (slug, fye) in get_ticker_map().items():
#     try:
#         analyze_company(t, slug, fye)
#     except Exception as e:
#         print(f"⚠ {t} 에러: {e}")

In [ ]:
import os, glob, re
from pathlib import Path
from datetime import date
import pandas as pd
import numpy as np

# ====== DCF 공통 가정 ======
WACC = 0.10
COE = 0.10
G_SHORT = 0.05
PROJ_YEARS = 5
G_TERM = 0.025

# ====== 경로 ======
BASE = Path(r"C:\Users\seung\OneDrive\주식")
BACKTEST_DIR = BASE / "Back Test"
FIN_DIR = BASE / "Financial_Data_real"
SUMMARY_DIR = FIN_DIR / "Summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = Path(r"C:\Users\seung\OneDrive\주식\Processed Data")

# ====== 유틸 ======
def canonical_company_display(name_or_slug: str) -> str:
    """'crispr-therapeutics-ag' -> 'Crispr Therapeutics Ag'"""
    disp = name_or_slug.replace("-", " ").strip()
    disp = re.sub(r"\s+", " ", disp)
    return disp.title()

def find_processed_csv(display_name: str):
    """Processed Data에서 DisplayName 기반 지표포함 CSV(가장 최신 수정본) 검색"""
    patterns = [
        str(PROCESSED_DIR / f"{display_name}_지표포함*.csv"),
        str(PROCESSED_DIR / f"{display_name}*_지표포함*.csv"),
    ]
    for pat in patterns:
        hits = glob.glob(pat)
        if hits:
            hits.sort(key=os.path.getmtime, reverse=True)
            return hits[0]
    return None

def safe_div(n: pd.Series, d: pd.Series, eps: float = 1e-9) -> pd.Series:
    n = pd.to_numeric(n, errors="coerce")
    d = pd.to_numeric(d, errors="coerce")
    d_safe = d.where(d.abs() > eps)
    return n / d_safe

def dcf_equity_value_from_fcf(fcf0, net_debt, rate, g_short, years, g_term):
    if pd.isna(fcf0) or fcf0 <= 0:
        return np.nan
    pv = sum(fcf0 * (1 + g_short)**t / (1 + rate)**t for t in range(1, years+1))
    cf_N = fcf0 * (1 + g_short)**years
    tv = cf_N * (1 + g_term) / (rate - g_term)
    pv_tv = tv / (1 + rate)**years
    return pv + pv_tv - (net_debt if pd.notna(net_debt) else 0)

def pct_change_abs(series, periods=1):
    prev = series.shift(periods)
    return (series - prev) / prev.abs() * 100

def pick_series(df: pd.DataFrame, candidates):
    """후보 열 중 존재하는 첫 숫자 Series 반환 (없으면 None)"""
    for c in candidates:
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            if s.notna().any():
                return s
    return None

def get_num(row: pd.Series, col: str):
    """row[col]을 float로, 없거나 비수치면 NaN"""
    if col in row.index and pd.notna(row[col]):
        try:
            return float(row[col])
        except Exception:
            return np.nan
    return np.nan

# ====== 메인 분석 ======
def analyze_company(TICKER, COMPANY_SLUG, FY_END_MONTH, MARKET_PRICE_OVERRIDE=None):
    print(f"\n▶ 분석 시작: {TICKER} ({COMPANY_SLUG})")

    # --- 폴더/파일명 정규화 (슬러그 → Title Case) ---
    display_name = canonical_company_display(COMPANY_SLUG)  # 예: 'Crispr Therapeutics Ag'
    company_folder = BACKTEST_DIR / display_name
    price_pattern = str(company_folder / f"* {display_name} Historical Data.csv")

    price_candidates = glob.glob(price_pattern)
    if not price_candidates and company_folder.exists():
        # 폴백: 폴더 내 '* Historical Data.csv' 중 display_name 포함
        price_candidates = [
            str(p) for p in company_folder.glob("* Historical Data.csv")
            if display_name.lower() in p.name.lower()
        ]
    if not price_candidates:
        print(f"❌ 가격 CSV 없음: {display_name}")
        return
    PRICE_CSV = price_candidates[0]

    FIN_XLSX = FIN_DIR / f"{TICKER}_financials_Q.xlsx"
    if not FIN_XLSX.exists():
        print(f"❌ 재무 엑셀 없음: {TICKER}")
        return

    # 재무 시트 로드
    def load_financials(fp):
        df = lambda sh: (pd.read_excel(fp, sheet_name=sh, index_col=0)
                           .T.reset_index().rename(columns={"index": "Date"})
                           .assign(Date=lambda x: pd.to_datetime(x["Date"])))
        return df("Income Statement"), df("Balance Sheet"), df("Cash Flow Statement"), df("Key Financial Ratios")

    inc, bal, cf, rat = load_financials(FIN_XLSX)
    q = inc.merge(bal, on="Date", how="left").merge(cf, on="Date", how="left").merge(rat, on="Date", how="left")
    q.sort_values("Date", inplace=True)

    # 가격 매칭 + Full Price 보존
    price_full = pd.read_csv(PRICE_CSV)
    price_full["Date"] = pd.to_datetime(price_full["Date"])
    price_full.rename(columns=lambda c: c.strip(), inplace=True)
    price_full.sort_values("Date", inplace=True)
    # Price 컬럼 탐색(Price/Close 계열 지원)
    price_col = next(
        (c for c in price_full.columns if c.lower().strip() in ("price","close","adj close","adj_close","adjclose")),
        None
    )
    if price_col is None:
        price_col = next(c for c in price_full.columns if "Price" in c)  # 최후 폴백
    px = price_full[["Date", price_col]].rename(columns={price_col: "Price"})
    q = pd.merge_asof(q, px, on="Date", direction="backward")
    if MARKET_PRICE_OVERRIDE is not None:
        q["Price"] = float(MARKET_PRICE_OVERRIDE)

    # -------- 분기별 계산 --------
    eps_series = pick_series(q, ["EPS - Earnings Per Share", "Basic EPS"])
    shares_series = pick_series(q, ["Shares Outstanding", "Basic Shares Outstanding"])

    # EPS가 없으면 Net Income / Shares로 대체
    if eps_series is None:
        if ("Net Income" in q.columns) and (shares_series is not None):
            eps_series = pd.to_numeric(q["Net Income"], errors="coerce") / shares_series
        else:
            eps_series = pd.Series(np.nan, index=q.index)

    price_series = pd.to_numeric(q["Price"], errors="coerce")
    div_total_q = pd.to_numeric(q.get("Common Stock Dividends Paid", np.nan), errors="coerce").abs()
    div_ps_q = safe_div(div_total_q, shares_series)
    div_yield_q = safe_div(div_ps_q, price_series) * 100

    liab = pd.to_numeric(q.get("Total Liabilities", np.nan), errors="coerce")
    cash = pd.to_numeric(q.get("Cash On Hand", np.nan), errors="coerce")
    ev_q = (price_series * shares_series) + liab - cash
    ebit_q = pd.to_numeric(q.get("EBIT", np.nan), errors="coerce")
    ev_ebit_q = safe_div(ev_q, ebit_q)

    def calc_q(r):
        pr  = get_num(r, "Price")
        sh  = get_num(r, "Shares Outstanding")
        if np.isnan(sh):
            sh = get_num(r, "Basic Shares Outstanding")
        tca = get_num(r, "Total Current Assets")
        tl  = get_num(r, "Total Liabilities")
        she = get_num(r, "Share Holder Equity")
        ni  = get_num(r, "Net Income")
        ebit= get_num(r, "EBIT")
        dep = get_num(r, "Total Depreciation And Amortization - Cash Flow")
        capx= get_num(r, "Net Change In Property, Plant, And Equipment")
        dwc = get_num(r, "Total Change In Assets/Liabilities")
        div = abs(get_num(r, "Common Stock Dividends Paid"))

        ncav = (tca - tl) if (pd.notna(tca) and pd.notna(tl)) else np.nan
        eps  = (ni / sh) if (pd.notna(ni) and pd.notna(sh) and sh != 0) else np.nan
        graham = np.sqrt(22.5 * max(eps, 0) * max((she / sh) if (pd.notna(she) and pd.notna(sh) and sh != 0) else 0, 0)) \
                 if pd.notna(sh) else np.nan

        fcff = (ebit*0.75 + dep - (-capx) - (-dwc)) if pd.notna(ebit) else np.nan
        owner = (ni + dep + capx + dwc) if pd.notna(ni) else np.nan
        ddm = (div * (1 + G_SHORT) / (COE - G_SHORT)) if pd.notna(div) and div > 0 else np.nan

        mkcap = pr * sh if (pd.notna(pr) and pd.notna(sh)) else np.nan
        underv_ncav = (mkcap < ncav) if (pd.notna(mkcap) and pd.notna(ncav)) else np.nan

        return pd.Series({
            "Price": pr,
            "Market Cap": mkcap,
            "NCAV": ncav,
            "Undervalued by NCAV": underv_ncav,
            "Graham Number": graham,
            "FCFF": fcff,
            "Owner Earnings": owner,
            "DDM Value": ddm
        })

    # 기본 분기 테이블
    quarterly = pd.concat([q[["Date"]], q.apply(calc_q, axis=1)], axis=1).set_index("Date")

    # === QoQ 핵심 지표 ===
    quarterly["EPS_Q"] = eps_series.values
    eps_q_growth_qoq = (eps_series - eps_series.shift(1)) / eps_series.shift(1).abs() * 100
    pe_q = safe_div(price_series, eps_series)
    peg_q = safe_div(pe_q, eps_q_growth_qoq)
    pegy_q = safe_div(pe_q, eps_q_growth_qoq + div_yield_q)

    quarterly["EPS_Growth_QoQ_%"] = eps_q_growth_qoq.values
    quarterly["PE_Q"] = pe_q.values
    quarterly["Div_Yield_Q_%"] = div_yield_q.values
    quarterly["PEG_Q"] = peg_q.values
    quarterly["PEGY_Q"] = pegy_q.values
    quarterly["EV/EBIT_Q"] = ev_ebit_q.values

    cols_order = [
        "Price", "Market Cap", "NCAV", "Undervalued by NCAV",
        "EPS_Q", "EPS_Growth_QoQ_%", "PE_Q", "Div_Yield_Q_%", "PEG_Q", "PEGY_Q", "EV/EBIT_Q",
        "Graham Number", "FCFF", "Owner Earnings", "DDM Value"
    ]
    quarterly = quarterly[[c for c in cols_order if c in quarterly.columns]]

    # -------- TTM 계산 --------
    qq = q.set_index("Date").sort_index()
    ttm = lambda col: qq[col].rolling(4, min_periods=4).sum()
    net = ttm("Net Income")
    ebit = ttm("EBIT")
    dep = ttm("Total Depreciation And Amortization - Cash Flow")
    capex = (-qq["Net Change In Property, Plant, And Equipment"]).rolling(4).sum()
    delta_wc = (-qq["Total Change In Assets/Liabilities"]).rolling(4).sum()
    div_ps = (abs(qq["Common Stock Dividends Paid"]) / qq["Shares Outstanding"]).rolling(4).sum()
    snap = qq[["Total Liabilities", "Cash On Hand", "Shares Outstanding",
               "Total Current Assets", "Total Assets", "Price"]]

    ncav_ttm = snap["Total Current Assets"] - snap["Total Liabilities"]
    market_cap_ttm = snap["Price"] * snap["Shares Outstanding"]
    roic = ebit / (snap["Total Current Assets"] + (snap["Total Assets"] - snap["Total Current Assets"]))
    ev = market_cap_ttm + snap["Total Liabilities"] - snap["Cash On Hand"]
    ev_ebit = ev / ebit

    eps_ttm = net / snap["Shares Outstanding"]
    pe_ttm = snap["Price"] / eps_ttm
    eps_yoy = pct_change_abs(eps_ttm, periods=4)
    div_yield_ttm = div_ps / snap["Price"] * 100
    peg_ttm = safe_div(pe_ttm, eps_yoy)
    pegy_ttm = safe_div(pe_ttm, eps_yoy + div_yield_ttm)

    # DCF
    net_debt = snap["Total Liabilities"] - snap["Cash On Hand"]
    dcf_fcff = pd.Series({
        dt: dcf_equity_value_from_fcf(
            (ebit.loc[dt] * 0.75 + dep.loc[dt] - capex.loc[dt] - delta_wc.loc[dt]),
            net_debt.loc[dt], WACC, G_SHORT, PROJ_YEARS, G_TERM
        ) for dt in snap.index
    })
    dcf_owner = pd.Series({
        dt: dcf_equity_value_from_fcf(
            (net.loc[dt] + dep.loc[dt] - capex.loc[dt] - delta_wc.loc[dt]),
            0.0, COE, G_SHORT, PROJ_YEARS, G_TERM
        ) for dt in snap.index
    })
    dcf_fcff_price = dcf_fcff / snap["Shares Outstanding"]
    dcf_owner_price = dcf_owner / snap["Shares Outstanding"]

    yearly = pd.DataFrame({
        "Market Cap": market_cap_ttm,
        "NCAV_TTM": ncav_ttm,
        "Undervalued by NCAV": market_cap_ttm < ncav_ttm,
        "ROIC": roic,
        "EV/EBIT": ev_ebit,
        "EPS_TTM": eps_ttm, "PE_TTM": pe_ttm, "EPS_Growth_YoY_%": eps_yoy,
        "PEG_TTM": peg_ttm, "PEGY_TTM": pegy_ttm,
        "Graham Number": quarterly["Graham Number"],
        "DDM Value": quarterly["DDM Value"],
        "DCF_FCFF_Price": dcf_fcff_price,
        "DCF_OwnerEarnings_Price": dcf_owner_price,
        "Altman Z": (1.2 * ((snap["Total Current Assets"] - snap["Total Liabilities"]) / snap["Total Assets"]) +
                     1.4 * ((qq["Retained Earnings (Accumulated Deficit)"]) / snap["Total Assets"]) +
                     3.3 * (ebit / snap["Total Assets"]) +
                     0.6 * (market_cap_ttm / snap["Total Liabilities"]) +
                     1.0 * (net / snap["Total Assets"]))
    })

    # 연말 스냅샷 (모든 월 지원)
    fq_map = {
        1:'Q-JAN', 2:'Q-FEB', 3:'Q-MAR', 4:'Q-APR', 5:'Q-MAY', 6:'Q-JUN',
        7:'Q-JUL', 8:'Q-AUG', 9:'Q-SEP', 10:'Q-OCT', 11:'Q-NOV', 12:'Q-DEC'
    }
    fq = yearly.index.to_period(fq_map.get(FY_END_MONTH, 'Q-DEC')).quarter
    snapshot = yearly[fq == 4]

    # ── Price_History: Processed Data의 지표포함 CSV 사용(없으면 원본 CSV 폴백) ──
    processed_path = find_processed_csv(display_name)
    if processed_path:
        price_hist_df = pd.read_csv(processed_path, encoding="utf-8-sig")
        if '날짜' in price_hist_df.columns:
            price_hist_df['날짜'] = pd.to_datetime(price_hist_df['날짜'], errors='coerce')
            price_hist_df = price_hist_df.sort_values('날짜')
    else:
        print(f"⚠ 가공본 없음: {display_name} → Price_History는 원본 CSV로 대체")
        price_hist_df = price_full  # 폴백

    # ── 저장 경로(out) 정의 ──
    out = SUMMARY_DIR / f"{date.today().isoformat()}_{TICKER}_{COMPANY_SLUG}_stock_summary.xlsx"

    # 저장 (Quarterly + Yearly_TTM + Inputs + Price_History)
    with pd.ExcelWriter(out, engine="xlsxwriter") as w:
        quarterly.round(4).to_excel(w, sheet_name="Quarterly")
        snapshot.round(4).to_excel(w, sheet_name="Yearly_TTM")
        q.to_excel(w, sheet_name="Inputs", index=False)
        price_hist_df.to_excel(w, sheet_name="Price_History", index=False)

    print(f"✅ 저장됨: {out}")

# ========== 전체 실행 ==========
# 외부에 정의된 get_ticker_map() 사용 가정
for t, (slug, fye) in get_ticker_map().items():
    try:
        analyze_company(t, slug, fye)
    except Exception as e:
        print(f"⚠ {t} 에러: {e}")


# Visualizaition

In [ ]:
# import pandas as pd
# import numpy as np  # ⬅ 추가
# import plotly.graph_objects as go
# import plotly.io as pio
# from pathlib import Path

# # 📁 경로 설정
# DATA_DIR = Path(r"C:\Users\seung\OneDrive\주식\Financial_Data_real\Summary\test")
# VISUAL_BASE = Path(r"C:\Users\seung\OneDrive\주식\Financial_Data_real\Summary\Visual")

# # 📊 그래프 저장 함수
# def save_plot(fig, filename, out_dir):
#     out_dir.mkdir(parents=True, exist_ok=True)
#     pio.write_html(fig, file=str(out_dir / f"{filename}.html"), auto_open=False)

#     # 📌 공통 레이아웃 구성 함수
# def apply_interactive_layout(fig, title, y1_title="Y1", y2_title=None, y3_title=None):
#     layout = dict(
#         title=title,
#         xaxis=dict(
#             title="Date",
#             rangeslider=dict(visible=True),
#             rangeselector=dict(
#                 buttons=list([
#                     dict(count=1, label="1m", step="month", stepmode="backward"),
#                     dict(count=3, label="3m", step="month", stepmode="backward"),
#                     dict(count=6, label="6m", step="month", stepmode="backward"),
#                     dict(count=1, label="YTD", step="year", stepmode="todate"),
#                     dict(count=1, label="1y", step="year", stepmode="backward"),
#                     dict(step="all")
#                 ])
#             )
#         ),
#         yaxis=dict(title=y1_title, side="left", autorange=True)
#     )
#     if y2_title:
#         layout["yaxis2"] = dict(title=y2_title, overlaying="y", side="right", autorange=True)
#     if y3_title:
#         layout["yaxis3"] = dict(title=y3_title, overlaying="y", side="right", position=0.95, anchor="free", autorange=True)

#     fig.update_layout(**layout)


# # 🔁 모든 파일 처리
# for file_path in DATA_DIR.glob("*.xlsx"):
#     stock_name = file_path.stem.split("_")[1]  # 예: 2025-08-28_GOOG_alphabet_stock_summary.xlsx → GOOG
#     out_dir = VISUAL_BASE / stock_name

#     try:
#         xls = pd.ExcelFile(file_path)
#         df_q = pd.read_excel(xls, sheet_name="Quarterly")
#         df_price = pd.read_excel(xls, sheet_name="Price_History")
#         df_y = pd.read_excel(xls, sheet_name="Yearly_TTM", index_col=0)

#         # Yearly_TTM 인덱스(Date) 복구
#         df_y.index = pd.to_datetime(df_y.index)
#         df_y.reset_index(inplace=True)
#         df_y.rename(columns={"index": "Date"}, inplace=True)

#         # 날짜 컬럼 정규화(한글 대비) + 정렬
#         for df in [df_q, df_price]:
#             if "Date" not in df.columns and "날짜" in df.columns:
#                 df.rename(columns={"날짜": "Date"}, inplace=True)
#             df["Date"] = pd.to_datetime(df["Date"])
#             df.sort_values("Date", inplace=True)

#         # 📊 그래프 1
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["EPS_TTM"], name="EPS_TTM", yaxis="y1"))
#         fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["EPS_Growth_YoY_%"], name="EPS Growth YoY%", yaxis="y2"))
#         fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y3", line=dict(dash='dash')))
#         fig.update_layout(title="Yearly: EPS, Growth YoY%, Price",
#                           xaxis=dict(title="Date"),
#                           yaxis=dict(title="EPS_TTM", side="left"),
#                           yaxis2=dict(title="EPS Growth", overlaying="y", side="right"),
#                           yaxis3=dict(title="Price", anchor="free", overlaying="y", side="right", position=0.95))
#         save_plot(fig, "01_eps_growth_price_yearly", out_dir)

#         # 📊 그래프 2
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["EPS_Q"], name="EPS_Q", yaxis="y1"))
#         fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["EPS_Growth_QoQ_%"], name="EPS Growth QoQ%", yaxis="y2"))
#         fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y3", line=dict(dash='dash')))
#         fig.update_layout(title="Quarterly: EPS, Growth QoQ%, Price",
#                           xaxis=dict(title="Date"),
#                           yaxis=dict(title="EPS_Q"),
#                           yaxis2=dict(title="QoQ Growth", overlaying="y", side="right"),
#                           yaxis3=dict(title="Price", anchor="free", overlaying="y", side="right", position=0.95))
#         save_plot(fig, "02_eps_growth_price_quarterly", out_dir)

#         # 📊 그래프 3
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["PEG_TTM"], name="PEG_TTM"))
#         fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dash")))
#         fig.update_layout(title="Yearly: PEG_TTM vs Price",
#                           xaxis_title="Date",
#                           yaxis=dict(title="PEG_TTM"),
#                           yaxis2=dict(title="Price", overlaying="y", side="right"))
#         save_plot(fig, "03_peg_price_yearly", out_dir)

#         # 📊 그래프 4
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["PEG_Q"], name="PEG_Q"))
#         fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dash")))
#         fig.update_layout(title="Quarterly: PEG_Q vs Price",
#                           xaxis_title="Date",
#                           yaxis=dict(title="PEG_Q"),
#                           yaxis2=dict(title="Price", overlaying="y", side="right"))
#         save_plot(fig, "04_peg_price_quarterly", out_dir)

#         # 📊 그래프 5
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["DCF_FCFF_Price"], name="DCF_FCFF"))
#         fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["DCF_OwnerEarnings_Price"], name="DCF_Owner"))
#         fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dot")))
#         fig.update_layout(title="Yearly: DCF Estimates vs Price",
#                           xaxis_title="Date",
#                           yaxis=dict(title="DCF Valuations"),
#                           yaxis2=dict(title="Price", overlaying="y", side="right"))
#         save_plot(fig, "05_dcf_price_yearly", out_dir)

#         # 📊 그래프 6 (Quarterly DCF가 있을 때만)
#         if "DCF_FCFF_Price" in df_q.columns and "DCF_OwnerEarnings_Price" in df_q.columns:
#             fig = go.Figure()
#             fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["DCF_FCFF_Price"], name="DCF_FCFF"))
#             fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["DCF_OwnerEarnings_Price"], name="DCF_Owner"))
#             fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dot")))
#             fig.update_layout(title="Quarterly: DCF Estimates vs Price",
#                               xaxis_title="Date",
#                               yaxis=dict(title="DCF Valuations"),
#                               yaxis2=dict(title="Price", overlaying="y", side="right"))
#             save_plot(fig, "06_dcf_price_quarterly", out_dir)

#         # 📊 그래프 7
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["Altman Z"], name="Altman Z"))
#         fig.update_layout(title="Yearly: Altman Z Score",
#                           xaxis_title="Date",
#                           yaxis_title="Z Score")
#         save_plot(fig, "07_altman_z", out_dir)

#         # 📊 그래프 8
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["PE_TTM"], name="PE_TTM"))
#         fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["ROIC"], name="ROIC"))
#         fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["EV/EBIT"], name="EV/EBIT"))
#         fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dash")))
#         fig.update_layout(title="Yearly: Valuation Multiples vs Price",
#                           xaxis_title="Date",
#                           yaxis=dict(title="Multiples"),
#                           yaxis2=dict(title="Price", overlaying="y", side="right"))
#         save_plot(fig, "08_multiples_price_yearly", out_dir)

#         # 📊 그래프 9
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["PE_Q"], name="PE_Q"))
#         if "ROIC" in df_q.columns:
#             fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["ROIC"], name="ROIC"))
#         fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["EV/EBIT_Q"], name="EV/EBIT_Q"))
#         fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dash")))
#         fig.update_layout(title="Quarterly: Valuation Multiples vs Price",
#                           xaxis_title="Date",
#                           yaxis=dict(title="Multiples"),
#                           yaxis2=dict(title="Price", overlaying="y", side="right"))
#         save_plot(fig, "09_multiples_price_quarterly", out_dir)

#         # ─────────────────────────────────────────────────────────
#         # 📊 그래프 10: Price (y1) vs (Volume / MarketCap) in % (y2)
#         # ─────────────────────────────────────────────────────────
#         def _parse_vol_series(df):
#             import re
#             if "Volume" in df.columns:
#                 s = pd.to_numeric(df["Volume"], errors="coerce")
#             elif "거래량" in df.columns:
#                 s = pd.to_numeric(df["거래량"], errors="coerce")
#             elif "Vol." in df.columns:
#                 def parse_str(x):
#                     if pd.isna(x): return np.nan
#                     s = str(x).strip().replace(",", "")
#                     m = 1
#                     if re.search(r"[Kk]$", s): m, s = 1_000, s[:-1]
#                     elif re.search(r"[Mm]$", s): m, s = 1_000_000, s[:-1]
#                     elif re.search(r"[Bb]$", s): m, s = 1_000_000_000, s[:-1]
#                     try: return float(s) * m
#                     except: return np.nan
#                 s = df["Vol."].map(parse_str)
#             else:
#                 s = pd.Series(index=df.index, dtype="float64")
#             return s

#         vol_daily = _parse_vol_series(df_price)

#         # Inputs 시트에서 Shares Outstanding 불러오기(우선 사용)
#         df_inputs = pd.read_excel(xls, sheet_name="Inputs")
#         if "Date" not in df_inputs.columns and "날짜" in df_inputs.columns:
#             df_inputs.rename(columns={"날짜": "Date"}, inplace=True)
#         df_inputs["Date"] = pd.to_datetime(df_inputs["Date"])
#         df_inputs.sort_values("Date", inplace=True)

#         so_col = None
#         for c in ["Shares Outstanding", "Basic Shares Outstanding"]:
#             if c in df_inputs.columns and pd.to_numeric(df_inputs[c], errors="coerce").notna().any():
#                 so_col = c
#                 break

#         # 가격 날짜 기준 프레임(반드시 오름차순)
#         price_dates = df_price[["Date"]].sort_values("Date").dropna()

#         if so_col:
#             so = pd.DataFrame({
#                 "Date": df_inputs["Date"],
#                 "SO": pd.to_numeric(df_inputs[so_col], errors="coerce")
#             }).dropna()
#             so.sort_values("Date", inplace=True)

#             so_daily = pd.merge_asof(price_dates, so, on="Date", direction="backward")
#             price_aligned = pd.merge(price_dates, df_price[["Date","Price"]], on="Date", how="left")["Price"]
#             mcap_daily = so_daily["SO"] * pd.to_numeric(price_aligned, errors="coerce")
#         else:
#             # 폴백: Yearly_TTM의 Market Cap을 asof로 전개
#             if "Market Cap" not in df_y.columns:
#                 print("  ⚠ Market Cap 소스 없음 → plot 10 스킵")
#                 mcap_daily = None
#             else:
#                 y_mcap = df_y[["Date", "Market Cap"]].dropna().sort_values("Date")
#                 mcap_daily = pd.merge_asof(price_dates, y_mcap, on="Date", direction="backward")["Market Cap"]
#                 price_aligned = pd.merge(price_dates, df_price[["Date","Price"]], on="Date", how="left")["Price"]

#         if mcap_daily is not None:
#             vol_aligned = pd.merge(price_dates, df_price[["Date"]].assign(Vol=vol_daily), on="Date", how="left")["Vol"]
#             ratio_pct = (vol_aligned / mcap_daily) * 100

#             fig = go.Figure()
#             fig.add_trace(go.Scatter(
#                 x=price_dates["Date"], y=price_aligned,
#                 name="Price", yaxis="y1"
#             ))
#             fig.add_trace(go.Scatter(
#                 x=price_dates["Date"], y=ratio_pct,
#                 name="Volume / Market Cap (%)", yaxis="y2", line=dict(dash="dash")
#             ))
#             fig.update_layout(
#                 title="Price vs Volume / Market Cap (%)",
#                 xaxis=dict(title="Date"),
#                 yaxis=dict(title="Price", side="left"),
#                 yaxis2=dict(title="Volume/MarketCap (%)", overlaying="y", side="right")
#             )
#             save_plot(fig, "10_volume_over_marketcap", out_dir)

#         print(f"✅ {stock_name} 완료")

#     except Exception as e:
#         print(f"❌ {file_path.name} 처리 중 오류 발생:", e)


In [ ]:
# Visualization UPdate

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# 📁 경로 설정
DATA_DIR = Path(r"C:\Users\seung\OneDrive\주식\Financial_Data_real\Summary\test")
VISUAL_BASE = Path(r"C:\Users\seung\OneDrive\주식\Financial_Data_real\Summary\Visual")

# 📊 그래프 저장 함수
def save_plot(fig, filename, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    pio.write_html(fig, file=str(out_dir / f"{filename}.html"), auto_open=False)

def apply_interactive_layout(fig, title, y1_title="Y1", y2_title=None, y3_title=None):
    layout = dict(
        title=title,
        xaxis=dict(
            title="Date",
            rangeslider=dict(visible=True),
            rangeselector=dict(
                buttons=[
                    dict(count=1, label="1m", step="month", stepmode="backward"),
                    dict(count=3, label="3m", step="month", stepmode="backward"),
                    dict(count=6, label="6m", step="month", stepmode="backward"),
                    dict(count=1, label="YTD", step="year", stepmode="todate"),
                    dict(count=1, label="1y", step="year", stepmode="backward"),
                    dict(step="all")
                ]
            )
        ),
        yaxis=dict(title=y1_title, side="left", anchor='x', autorange=True, fixedrange=False)
    )
    if y2_title:
        layout["yaxis2"] = dict(title=y2_title, overlaying="y", side="right", anchor='x', autorange=True, fixedrange=False)
    if y3_title:
        layout["yaxis3"] = dict(title=y3_title, overlaying="y", side="right", position=0.95, anchor='x', autorange=True, fixedrange=False)

    fig.update_layout(**layout)


# 🔁 모든 파일 처리
for file_path in DATA_DIR.glob("*.xlsx"):
    stock_name = file_path.stem.split("_")[1]
    out_dir = VISUAL_BASE / stock_name

    try:
        xls = pd.ExcelFile(file_path)
        df_q = pd.read_excel(xls, sheet_name="Quarterly")
        df_price = pd.read_excel(xls, sheet_name="Price_History")
        df_y = pd.read_excel(xls, sheet_name="Yearly_TTM", index_col=0)

        df_y.index = pd.to_datetime(df_y.index)
        df_y.reset_index(inplace=True)
        df_y.rename(columns={"index": "Date"}, inplace=True)

        for df in [df_q, df_price]:
            if "Date" not in df.columns and "날짜" in df.columns:
                df.rename(columns={"날짜": "Date"}, inplace=True)
            df["Date"] = pd.to_datetime(df["Date"])
            df.sort_values("Date", inplace=True)

        # 그래프 1
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["EPS_TTM"], name="EPS_TTM", yaxis="y1"))
        fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["EPS_Growth_YoY_%"], name="EPS Growth YoY%", yaxis="y2"))
        fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y3", line=dict(dash='dash')))
        apply_interactive_layout(fig, "Yearly: EPS, Growth YoY%, Price", "EPS_TTM", "EPS Growth", "Price")
        save_plot(fig, "01_eps_growth_price_yearly", out_dir)

        # 그래프 2
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["EPS_Q"], name="EPS_Q", yaxis="y1"))
        fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["EPS_Growth_QoQ_%"], name="EPS Growth QoQ%", yaxis="y2"))
        fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y3", line=dict(dash='dash')))
        apply_interactive_layout(fig, "Quarterly: EPS, Growth QoQ%, Price", "EPS_Q", "QoQ Growth", "Price")
        save_plot(fig, "02_eps_growth_price_quarterly", out_dir)

        # 그래프 3
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["PEG_TTM"], name="PEG_TTM"))
        fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dash")))
        apply_interactive_layout(fig, "Yearly: PEG_TTM vs Price", "PEG_TTM", "Price")
        save_plot(fig, "03_peg_price_yearly", out_dir)

        # 그래프 4
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["PEG_Q"], name="PEG_Q"))
        fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dash")))
        apply_interactive_layout(fig, "Quarterly: PEG_Q vs Price", "PEG_Q", "Price")
        save_plot(fig, "04_peg_price_quarterly", out_dir)

        # 그래프 5
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["DCF_FCFF_Price"], name="DCF_FCFF"))
        fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["DCF_OwnerEarnings_Price"], name="DCF_Owner"))
        fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dot")))
        apply_interactive_layout(fig, "Yearly: DCF Estimates vs Price", "DCF Valuations", "Price")
        save_plot(fig, "05_dcf_price_yearly", out_dir)

        # 그래프 6
        if "DCF_FCFF_Price" in df_q.columns and "DCF_OwnerEarnings_Price" in df_q.columns:
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["DCF_FCFF_Price"], name="DCF_FCFF"))
            fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["DCF_OwnerEarnings_Price"], name="DCF_Owner"))
            fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dot")))
            apply_interactive_layout(fig, "Quarterly: DCF Estimates vs Price", "DCF Valuations", "Price")
            save_plot(fig, "06_dcf_price_quarterly", out_dir)

        # 그래프 7
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["Altman Z"], name="Altman Z"))
        apply_interactive_layout(fig, "Yearly: Altman Z Score", "Z Score")
        save_plot(fig, "07_altman_z", out_dir)

        # 그래프 8
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["PE_TTM"], name="PE_TTM"))
        fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["ROIC"], name="ROIC"))
        fig.add_trace(go.Scatter(x=df_y["Date"], y=df_y["EV/EBIT"], name="EV/EBIT"))
        fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dash")))
        apply_interactive_layout(fig, "Yearly: Valuation Multiples vs Price", "Multiples", "Price")
        save_plot(fig, "08_multiples_price_yearly", out_dir)

        # 그래프 9
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["PE_Q"], name="PE_Q"))
        if "ROIC" in df_q.columns:
            fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["ROIC"], name="ROIC"))
        fig.add_trace(go.Scatter(x=df_q["Date"], y=df_q["EV/EBIT_Q"], name="EV/EBIT_Q"))
        fig.add_trace(go.Scatter(x=df_price["Date"], y=df_price["Price"], name="Price", yaxis="y2", line=dict(dash="dash")))
        apply_interactive_layout(fig, "Quarterly: Valuation Multiples vs Price", "Multiples", "Price")
        save_plot(fig, "09_multiples_price_quarterly", out_dir)

        # ───────────── 그래프 10: Price vs Volume / Market Cap (%) ─────────────
        def _parse_vol_series(df):
            import re
            if "Volume" in df.columns:
                s = pd.to_numeric(df["Volume"], errors="coerce")
            elif "거래량" in df.columns:
                s = pd.to_numeric(df["거래량"], errors="coerce")
            elif "Vol." in df.columns:
                def parse_str(x):
                    if pd.isna(x): return np.nan
                    s = str(x).strip().replace(",", "")
                    m = 1
                    if re.search(r"[Kk]$", s): m, s = 1_000, s[:-1]
                    elif re.search(r"[Mm]$", s): m, s = 1_000_000, s[:-1]
                    elif re.search(r"[Bb]$", s): m, s = 1_000_000_000, s[:-1]
                    try: return float(s) * m
                    except: return np.nan
                s = df["Vol."].map(parse_str)
            else:
                s = pd.Series(index=df.index, dtype="float64")
            return s

        vol_daily = _parse_vol_series(df_price)
        df_inputs = pd.read_excel(xls, sheet_name="Inputs")
        if "Date" not in df_inputs.columns and "날짜" in df_inputs.columns:
            df_inputs.rename(columns={"날짜": "Date"}, inplace=True)
        df_inputs["Date"] = pd.to_datetime(df_inputs["Date"])
        df_inputs.sort_values("Date", inplace=True)

        so_col = None
        for c in ["Shares Outstanding", "Basic Shares Outstanding"]:
            if c in df_inputs.columns and pd.to_numeric(df_inputs[c], errors="coerce").notna().any():
                so_col = c
                break

        price_dates = df_price[["Date"]].sort_values("Date").dropna()

        if so_col:
            so = pd.DataFrame({
                "Date": df_inputs["Date"],
                # 📌 Million 단위 → 주식 수로 변환
                "SO": pd.to_numeric(df_inputs[so_col], errors="coerce") * 1_000_000
            }).dropna()
            so.sort_values("Date", inplace=True)

            so_daily = pd.merge_asof(price_dates, so, on="Date", direction="backward")
            price_aligned = pd.merge(price_dates, df_price[["Date", "Price"]], on="Date", how="left")["Price"]
            mcap_daily = so_daily["SO"] * pd.to_numeric(price_aligned, errors="coerce")
        else:
            if "Market Cap" not in df_y.columns:
                print("  ⚠ Market Cap 소스 없음 → plot 10 스킵")
                mcap_daily = None
            else:
                y_mcap = df_y[["Date", "Market Cap"]].dropna().sort_values("Date")
                mcap_daily = pd.merge_asof(price_dates, y_mcap, on="Date", direction="backward")["Market Cap"]
                price_aligned = pd.merge(price_dates, df_price[["Date", "Price"]], on="Date", how="left")["Price"]

        if mcap_daily is not None:
            vol_aligned = pd.merge(price_dates, df_price[["Date"]].assign(Vol=vol_daily), on="Date", how="left")["Vol"]
            ratio_pct = (vol_aligned / mcap_daily) * 100

            # 🔍 디버깅용 출력 (최근 5개 샘플)
            print("\n📌 Volume / Market Cap 계산 예시:")
            print(pd.DataFrame({
                "Date": price_dates["Date"],
                "Price": price_aligned,
                "Volume": vol_aligned,
                "SO": so_daily["SO"] if so_col else np.nan,
                "Market Cap": mcap_daily,
                "Vol/MktCap%": ratio_pct
            }).tail(5))

            fig = go.Figure()
            fig.add_trace(go.Scatter(x=price_dates["Date"], y=price_aligned, name="Price", yaxis="y1"))
            fig.add_trace(go.Scatter(x=price_dates["Date"], y=ratio_pct, name="Volume / Market Cap (%)", yaxis="y2", line=dict(dash="dash")))
            apply_interactive_layout(fig, "Price vs Volume / Market Cap (%)", "Price", "Volume/MarketCap (%)")
            save_plot(fig, "10_volume_over_marketcap", out_dir)

            save_plot(fig, "10_volume_over_marketcap", out_dir)

        print(f"✅ {stock_name} 완료")

    except Exception as e:
        print(f"❌ {file_path.name} 처리 중 오류 발생:", e)

        # 📁 Excel 파일로 저장할 DataFrame 구성
export_df = pd.DataFrame({
    "Date": price_dates["Date"],
    "Price": price_aligned,
    "Volume": vol_aligned,
    "Shares Outstanding": so_daily["SO"] if so_col else np.nan,
    "Market Cap": mcap_daily,
    "Vol/MktCap (%)": ratio_pct
})

# 📄 Excel로 저장
export_path = out_dir / "volume_marketcap_analysis.xlsx"
export_df.to_excel(export_path, index=False)

print(f"📁 Volume/MarketCap 데이터 저장 완료: {export_path}")


## Transformer

In [ ]:
import pandas as pd
import numpy as np
import talib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import os
import time

# ─────────────────────────────
# 1. 경로 설정
# ─────────────────────────────
print("🔧 Stage 1: Setting paths...")
SUMMARY_FILE = Path(r"C:\Users\seung\OneDrive\주식\Financial_Data_real\Summary\2025-08-30_NVDA_nvidia_stock_summary.xlsx")
MACRO_DIR = Path(r"C:\Users\seung\OneDrive\주식\Macro Data")
RESULT_DIR = Path(r"C:\Users\seung\OneDrive\주식\Transformer Result")

os.makedirs(RESULT_DIR, exist_ok=True)

# ─────────────────────────────
# 2. 회사 데이터 불러오기
# ─────────────────────────────
print("📂 Stage 2: Loading company data...")
price_df = pd.read_excel(SUMMARY_FILE, sheet_name="Price_History", parse_dates=["Date"])
quarterly_df = pd.read_excel(SUMMARY_FILE, sheet_name="Quarterly", parse_dates=["Date"])
inputs_df = pd.read_excel(SUMMARY_FILE, sheet_name="Inputs", parse_dates=["Date"])

print(f"   - Price data shape: {price_df.shape}")
print(f"   - Quarterly data shape: {quarterly_df.shape}")
print(f"   - Inputs data shape: {inputs_df.shape}")

price_df = price_df.set_index("Date").sort_index()
quarterly_df = quarterly_df.set_index("Date").sort_index().resample("D").ffill()
inputs_df = inputs_df.set_index("Date").sort_index().resample("D").ffill()

company_df = price_df.join([quarterly_df, inputs_df], how="left").ffill()
print(f"   - Combined company_df shape: {company_df.shape}")

# ─────────────────────────────
# 3. 기술적 지표 추가
# ─────────────────────────────
print("📊 Stage 3: Adding technical indicators...")
close = company_df["종가"].values
high = company_df["고가"].values
low = company_df["저가"].values
volume = company_df["거래량"].values

company_df["RSI_14"] = talib.RSI(close, timeperiod=14)
company_df["MACD"], company_df["MACD_SIG"], company_df["MACD_HIST"] = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
company_df["BB_UPPER"], company_df["BB_MID"], company_df["BB_LOWER"] = talib.BBANDS(close, timeperiod=20)
company_df["MA20"] = talib.SMA(close, timeperiod=20)
company_df["MA50"] = talib.SMA(close, timeperiod=50)
company_df["MA200"] = talib.SMA(close, timeperiod=200)
company_df["OBV"] = talib.OBV(close, volume)

print("   - Technical indicators added!")

# ─────────────────────────────
# 4. 매크로 데이터 불러오기
# ─────────────────────────────
print("🌍 Stage 4: Loading macro data...")
macro_files = {
    "CPI": "1970.01.01_2025.08.01 CPI.csv",
    "FedFunds": "1970.01.01_2025.08.01 FedFundsRate.csv",
    "Unemployment": "1970.01.01_2025.08.01 Unemployment.csv",
    "YieldCurve": "1970.01.01_2025.08.01 YieldCurve.csv",
    "WTI": "1986.01.01_2025.08.01 WTI.csv",
    "VIX": "1990.01.02_2025.09.18 VIX.csv",
    "HY_Spread": "1996.12.31_2025.09.18 HY_Spread.csv"
}

macro_dfs = []
for name, file in macro_files.items():
    print(f"   - Loading {name}...")
    df = pd.read_csv(MACRO_DIR / file, parse_dates=["Date"])
    df = df.set_index("Date").sort_index().resample("D").ffill()
    df = df.rename(columns={"Value": name})
    macro_dfs.append(df)

macro_df = pd.concat(macro_dfs, axis=1)
print(f"   - Macro data shape: {macro_df.shape}")

# ─────────────────────────────
# 5. Feature + Label 구성
# ─────────────────────────────
print("🛠️ Stage 5: Building features and labels...")
all_features = company_df.join(macro_df, how="left").ffill()
all_features["Return_5d"] = company_df["종가"].pct_change(5).shift(-5)
all_features["Target"] = (all_features["Return_5d"] > 0).astype(int)

SEQ_LEN = 60
features = all_features.drop(columns=["Return_5d", "Target"], errors="ignore")

# 🩹 Timestamp 제거 + float32 변환
for col in features.columns:
    if np.issubdtype(features[col].dtype, np.datetime64):
        features = features.drop(columns=[col])

features = features.values.astype(np.float32)
labels = all_features["Target"].values.astype(np.float32)

X, y = [], []
for i in range(len(features) - SEQ_LEN):
    X.append(features[i:i+SEQ_LEN])
    y.append(labels[i+SEQ_LEN])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f"   - X shape: {X.shape}")
print(f"   - y shape: {y.shape}")

# ─────────────────────────────
# 6. Transformer 모델
# ─────────────────────────────
print("🤖 Stage 6: Building Transformer model...")
def build_transformer_model(seq_len, n_features, d_model=64, n_heads=4, ff_dim=128, n_layers=2):
    inputs = layers.Input(shape=(seq_len, n_features))
    x = layers.Dense(d_model)(inputs)

    for _ in range(n_layers):
        attn_out = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model)(x, x)
        attn_out = layers.Dropout(0.1)(attn_out)
        x = layers.LayerNormalization()(x + attn_out)

        ff_out = layers.Dense(ff_dim, activation="relu")(x)
        ff_out = layers.Dense(d_model)(ff_out)
        ff_out = layers.Dropout(0.1)(ff_out)
        x = layers.LayerNormalization()(x + ff_out)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-4),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

model = build_transformer_model(SEQ_LEN, X.shape[2])
model.summary()

# ─────────────────────────────
# 7. 학습 + 결과 저장
# ─────────────────────────────
print("🚀 Stage 7: Training started... (progress shown below)")
start_time = time.time()

history = model.fit(
    X, y,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

print(f"✅ Training finished in {(time.time()-start_time):.2f} sec")

print("💾 Saving model and results...")
model.save(RESULT_DIR / "transformer_model.h5")

hist_df = pd.DataFrame(history.history)
hist_df.to_csv(RESULT_DIR / "training_log.csv", index=False)

y_pred = (model.predict(X) > 0.5).astype(int)
pred_df = pd.DataFrame({
    "Date": all_features.index[SEQ_LEN:],
    "True": y,
    "Pred": y_pred.flatten()
})
pred_df.to_csv(RESULT_DIR / "predictions.csv", index=False)

print("🎉 All results saved in:", RESULT_DIR)


In [ ]:
import pandas as pd
import numpy as np
import talib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import os, time
from sklearn.preprocessing import StandardScaler

# ─────────────────────────────
# 1. 경로 설정
# ─────────────────────────────
print("🔧 Stage 1: Setting paths...")
SUMMARY_FILE = Path(r"C:\Users\seung\OneDrive\주식\Financial_Data_real\Summary\2025-08-30_NVDA_nvidia_stock_summary.xlsx")
MACRO_DIR = Path(r"C:\Users\seung\OneDrive\주식\Macro Data")
RESULT_DIR = Path(r"C:\Users\seung\OneDrive\주식\Transformer Result")
os.makedirs(RESULT_DIR, exist_ok=True)

# ─────────────────────────────
# 2. 회사 데이터 불러오기
# ─────────────────────────────
print("📂 Stage 2: Loading company data...")
price_df = pd.read_excel(SUMMARY_FILE, sheet_name="Price_History")

# 첫 번째 컬럼을 Date로 사용
price_df = price_df.rename(columns={price_df.columns[0]: "Date"})
price_df = price_df.loc[:, ~price_df.columns.duplicated()]
price_df["Date"] = pd.to_datetime(price_df["Date"], errors="coerce")
price_df = price_df.set_index("Date").sort_index()

# Quarterly / Inputs
quarterly_df = pd.read_excel(SUMMARY_FILE, sheet_name="Quarterly", parse_dates=["Date"])
inputs_df = pd.read_excel(SUMMARY_FILE, sheet_name="Inputs", parse_dates=["Date"])
quarterly_df = quarterly_df.set_index("Date").sort_index().resample("D").ffill()
inputs_df = inputs_df.set_index("Date").sort_index().resample("D").ffill()

company_df = price_df.join([quarterly_df, inputs_df], how="left").ffill()
print(f"   - company_df shape: {company_df.shape}")

# ─────────────────────────────
# 3. 기술적 지표 추가
# ─────────────────────────────
print("📊 Stage 3: Adding technical indicators...")
close, high, low, volume = company_df["종가"].values, company_df["고가"].values, company_df["저가"].values, company_df["거래량"].values
company_df["RSI_14"] = talib.RSI(close, timeperiod=14)
company_df["MACD"], company_df["MACD_SIG"], company_df["MACD_HIST"] = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
company_df["BB_UPPER"], company_df["BB_MID"], company_df["BB_LOWER"] = talib.BBANDS(close, timeperiod=20)
company_df["MA20"] = talib.SMA(close, timeperiod=20)
company_df["MA50"] = talib.SMA(close, timeperiod=50)
company_df["MA200"] = talib.SMA(close, timeperiod=200)
company_df["OBV"] = talib.OBV(close, volume)

# ─────────────────────────────
# 4. 매크로 데이터 불러오기
# ─────────────────────────────
print("🌍 Stage 4: Loading macro data...")
macro_files = {
    "CPI": "1970.01.01_2025.08.01 CPI.csv",
    "FedFunds": "1970.01.01_2025.08.01 FedFundsRate.csv",
    "Unemployment": "1970.01.01_2025.08.01 Unemployment.csv",
    "YieldCurve": "1970.01.01_2025.08.01 YieldCurve.csv",
    "WTI": "1986.01.01_2025.08.01 WTI.csv",
    "VIX": "1990.01.02_2025.09.18 VIX.csv",
    "HY_Spread": "1996.12.31_2025.09.18 HY_Spread.csv"
}
macro_dfs = []
for name, file in macro_files.items():
    print(f"   - Loading {name}...")
    df = pd.read_csv(MACRO_DIR / file, parse_dates=["Date"])
    df = df.set_index("Date").sort_index().resample("D").ffill()
    df = df.rename(columns={"Value": name})
    macro_dfs.append(df)
macro_df = pd.concat(macro_dfs, axis=1)
print(f"   - macro_df shape: {macro_df.shape}")

# ─────────────────────────────
# 5. CSV 저장
# ─────────────────────────────
print("💾 Saving intermediate DataFrames...")

# Date를 일반 컬럼으로 저장
company_df.reset_index().to_csv(RESULT_DIR / "company_df.csv", encoding="utf-8-sig", index=False)
macro_df.reset_index().to_csv(RESULT_DIR / "macro_df.csv", encoding="utf-8-sig", index=False)

print("🎉 Saved company_df.csv and macro_df.csv to:", RESULT_DIR)


# # ─────────────────────────────
# # 5. Feature + Label 구성 (5일 뒤 +3% 수익률만 1)
# # ─────────────────────────────
# print("🛠️ Stage 5: Building features and labels...")
# all_features = company_df.join(macro_df, how="left").ffill()
# all_features["Return_5d"] = company_df["종가"].pct_change(5).shift(-5)
# all_features["Target"] = (all_features["Return_5d"] > 0.03).astype(int)  # 3% 이상 상승만 1

# SEQ_LEN = 60
# features = all_features.drop(columns=["Return_5d", "Target"], errors="ignore")

# # Timestamp 제거 + scaling
# for col in features.columns:
#     if np.issubdtype(features[col].dtype, np.datetime64):
#         features = features.drop(columns=[col])
# scaler = StandardScaler()
# features = scaler.fit_transform(features.fillna(0))  # NaN 방지

# labels = all_features["Target"].values.astype(np.float32)

# X, y = [], []
# for i in range(len(features) - SEQ_LEN):
#     X.append(features[i:i+SEQ_LEN])
#     y.append(labels[i+SEQ_LEN])
# X = np.array(X, dtype=np.float32)
# y = np.array(y, dtype=np.float32)

# print(f"   - X shape: {X.shape}, y shape: {y.shape}, positive ratio: {y.mean():.3f}")

# # ─────────────────────────────
# # 6. Transformer 모델
# # ─────────────────────────────
# print("🤖 Stage 6: Building Transformer model...")
# def build_transformer_model(seq_len, n_features, d_model=64, n_heads=4, ff_dim=128, n_layers=2):
#     inputs = layers.Input(shape=(seq_len, n_features))
#     x = layers.Dense(d_model)(inputs)
#     for _ in range(n_layers):
#         attn_out = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model)(x, x)
#         attn_out = layers.Dropout(0.1)(attn_out)
#         x = layers.LayerNormalization()(x + attn_out)
#         ff_out = layers.Dense(ff_dim, activation="relu")(x)
#         ff_out = layers.Dense(d_model)(ff_out)
#         ff_out = layers.Dropout(0.1)(ff_out)
#         x = layers.LayerNormalization()(x + ff_out)
#     x = layers.GlobalAveragePooling1D()(x)
#     x = layers.Dense(64, activation="relu")(x)
#     x = layers.Dropout(0.2)(x)
#     outputs = layers.Dense(1, activation="sigmoid")(x)
#     model = keras.Model(inputs, outputs)
#     model.compile(optimizer=keras.optimizers.Adam(3e-4),
#                   loss="binary_crossentropy",
#                   metrics=["accuracy"])
#     return model

# model = build_transformer_model(SEQ_LEN, X.shape[2])
# model.summary()

# # ─────────────────────────────
# # 7. 학습 + 저장
# # ─────────────────────────────
# print("🚀 Stage 7: Training started...")
# from sklearn.utils.class_weight import compute_class_weight
# class_weights = compute_class_weight("balanced", classes=np.unique(y.astype(int)), y=y.astype(int))
# class_weights = {i: w for i, w in enumerate(class_weights)}

# es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

# history = model.fit(
#     X, y,
#     epochs=50,
#     batch_size=32,
#     validation_split=0.2,
#     class_weight=class_weights,
#     callbacks=[es],
#     verbose=1
# )

# print("✅ Training finished")

# # Save
# model.save(RESULT_DIR / "transformer_model.h5")
# hist_df = pd.DataFrame(history.history)
# hist_df.to_csv(RESULT_DIR / "training_log.csv", index=False)
# y_pred = (model.predict(X) > 0.5).astype(int)
# pd.DataFrame({"Date": all_features.index[SEQ_LEN:], "True": y, "Pred": y_pred.flatten()}).to_csv(RESULT_DIR / "predictions.csv", index=False)

# print("🎉 All results saved in:", RESULT_DIR)


In [ ]:
import matplotlib.pyplot as plt

# Save
model.save(RESULT_DIR / "transformer_model.h5")
hist_df = pd.DataFrame(history.history)
hist_df.to_csv(RESULT_DIR / "training_log.csv", index=False)

# 예측
y_pred = (model.predict(X) > 0.5).astype(int)
pred_df = pd.DataFrame({
    "Date": all_features.index[SEQ_LEN:],
    "True": y,
    "Pred": y_pred.flatten(),
    "Close": company_df["종가"].iloc[SEQ_LEN:].values  # 실제 종가 붙이기
})
pred_df.to_csv(RESULT_DIR / "predictions.csv", index=False)

print("🎉 All results saved in:", RESULT_DIR)

# ─────────────────────────────
# 8. 시각화
# ─────────────────────────────
print("📊 Plotting predictions vs stock price...")

plt.figure(figsize=(14, 6))
plt.plot(pred_df["Date"], pred_df["Close"], label="Stock Price", color="blue")

# 매수 시그널 (Pred==1) 표시
buy_signals = pred_df[pred_df["Pred"] == 1]
plt.scatter(buy_signals["Date"], buy_signals["Close"], label="Predicted Buy (≥+3% in 5d)", color="green", marker="^", alpha=0.8)

# 실제로 +3% 달성한 경우 (True==1)
true_signals = pred_df[pred_df["True"] == 1]
plt.scatter(true_signals["Date"], true_signals["Close"], label="Actual ≥+3% in 5d", color="red", marker="o", alpha=0.6)

plt.title("Predictions vs Stock Price (5d return ≥ +3%)")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(RESULT_DIR / "prediction_vs_price.png", dpi=200)
plt.show()


In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.23.5


In [ ]:
!pip install --upgrade tensorflow==2.14


In [ ]:
!pip install TA-Lib


In [ ]:
Kernel → Restart Kernel


In [ ]:
!pip uninstall -y protobuf


In [ ]:
!pip install --upgrade --force-reinstall protobuf==3.20.3


In [ ]:
!pip install --upgrade tensorflow==2.14


In [ ]:
!pip uninstall -y tensorflow protobuf numpy
!pip install numpy==1.23.5
!pip install protobuf==3.20.3
!pip install tensorflow==2.14


In [ ]:
!pip install tensorflow


In [ ]:
!pip install --upgrade numpy scipy scikit-learn


In [ ]:
!pip install --upgrade --force-reinstall numpy pandas scipy scikit-learn


In [ ]:
!pip install "numpy<2" --force-reinstall


In [ ]:
!pip install --upgrade openpyxl



In [ ]:
# 20일 주가 예측 transformer

In [ ]:
import pandas as pd
import numpy as np
import talib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import os
from sklearn.preprocessing import StandardScaler

# ─────────────────────────────
# 1. 데이터 로드 (전부 동일)
# ─────────────────────────────
SUMMARY_FILE = Path(r"C:\Users\seung\OneDrive\주식\Financial_Data_real\Summary\2025-08-30_NVDA_nvidia_stock_summary.xlsx")
MACRO_DIR = Path(r"C:\Users\seung\OneDrive\주식\Macro Data")
RESULT_DIR = Path(r"C:\Users\seung\OneDrive\주식\Transformer Result")
os.makedirs(RESULT_DIR, exist_ok=True)

price_df = pd.read_excel(SUMMARY_FILE, sheet_name="Price_History", parse_dates=["Date"])
quarterly_df = pd.read_excel(SUMMARY_FILE, sheet_name="Quarterly", parse_dates=["Date"])
inputs_df = pd.read_excel(SUMMARY_FILE, sheet_name="Inputs", parse_dates=["Date"])

price_df = price_df.set_index("Date").sort_index()
quarterly_df = quarterly_df.set_index("Date").sort_index().resample("D").ffill()
inputs_df = inputs_df.set_index("Date").sort_index().resample("D").ffill()
company_df = price_df.join([quarterly_df, inputs_df], how="left").ffill()

# ─────────────────────────────
# 2. 기술적 지표 + 매크로 (생략, 동일하게 추가)
# ─────────────────────────────
close, high, low, volume = company_df["종가"].values, company_df["고가"].values, company_df["저가"].values, company_df["거래량"].values
company_df["RSI_14"] = talib.RSI(close, timeperiod=14)
company_df["MACD"], company_df["MACD_SIG"], company_df["MACD_HIST"] = talib.MACD(close, 12, 26, 9)
company_df["BB_UPPER"], company_df["BB_MID"], company_df["BB_LOWER"] = talib.BBANDS(close, timeperiod=20)
company_df["MA20"] = talib.SMA(close, 20)
company_df["MA50"] = talib.SMA(close, 50)
company_df["MA200"] = talib.SMA(close, 200)
company_df["OBV"] = talib.OBV(close, volume)

macro_files = {
    "CPI": "1970.01.01_2025.08.01 CPI.csv",
    "FedFunds": "1970.01.01_2025.08.01 FedFundsRate.csv",
    "Unemployment": "1970.01.01_2025.08.01 Unemployment.csv",
    "YieldCurve": "1970.01.01_2025.08.01 YieldCurve.csv",
    "WTI": "1986.01.01_2025.08.01 WTI.csv",
    "VIX": "1990.01.02_2025.09.18 VIX.csv",
    "HY_Spread": "1996.12.31_2025.09.18 HY_Spread.csv"
}
macro_dfs = []
for name, file in macro_files.items():
    df = pd.read_csv(MACRO_DIR / file, parse_dates=["Date"])
    df = df.set_index("Date").sort_index().resample("D").ffill()
    df = df.rename(columns={"Value": name})
    macro_dfs.append(df)
macro_df = pd.concat(macro_dfs, axis=1)

# ─────────────────────────────
# 3. Feature + Label (수익률 회귀)
# ─────────────────────────────
all_features = company_df.join(macro_df, how="left").ffill()

# 5일 뒤, 20일 뒤 수익률 (연속값)
all_features["Return_5d"] = company_df["종가"].pct_change(5).shift(-5)
all_features["Return_20d"] = company_df["종가"].pct_change(20).shift(-20)

SEQ_LEN = 60
features = all_features.drop(columns=["Return_5d","Return_20d"], errors="ignore")

for col in features.columns:
    if np.issubdtype(features[col].dtype, np.datetime64):
        features = features.drop(columns=[col])
scaler = StandardScaler()
features = scaler.fit_transform(features.fillna(0))

labels_5d = all_features["Return_5d"].values.astype(np.float32)
labels_20d = all_features["Return_20d"].values.astype(np.float32)

X, y5, y20 = [], [], []
for i in range(len(features) - SEQ_LEN - 20):
    X.append(features[i:i+SEQ_LEN])
    y5.append(labels_5d[i+SEQ_LEN])
    y20.append(labels_20d[i+SEQ_LEN])
X = np.array(X, dtype=np.float32)
y5 = np.array(y5, dtype=np.float32)
y20 = np.array(y20, dtype=np.float32)

print(f"X shape: {X.shape}, y5 mean: {np.nanmean(y5):.3f}, y20 mean: {np.nanmean(y20):.3f}")

# ─────────────────────────────
# 4. Transformer (회귀)
# ─────────────────────────────
def build_transformer_regressor(seq_len, n_features, d_model=64, n_heads=4, ff_dim=128, n_layers=2):
    inputs = layers.Input(shape=(seq_len, n_features))
    x = layers.Dense(d_model)(inputs)
    for _ in range(n_layers):
        attn_out = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model)(x, x)
        attn_out = layers.Dropout(0.1)(attn_out)
        x = layers.LayerNormalization()(x + attn_out)
        ff_out = layers.Dense(ff_dim, activation="relu")(x)
        ff_out = layers.Dense(d_model)(ff_out)
        ff_out = layers.Dropout(0.1)(ff_out)
        x = layers.LayerNormalization()(x + ff_out)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation="linear")(x)  # ← 회귀
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(3e-4),
                  loss="mse",
                  metrics=["mae"])
    return model

model_20d = build_transformer_regressor(SEQ_LEN, X.shape[2])
model_20d.summary()

# ─────────────────────────────
# 5. Training + Custom Callback
# ─────────────────────────────
class ReturnEvalCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(X, verbose=0).flatten()
        top_idx = np.argsort(preds)[-int(len(preds)*0.1):]  # 상위 10%만 매수
        cum_return = np.nanmean(y20[top_idx])
        print(f"Epoch {epoch+1}: 📈 Top-10% 전략 평균 20d Return = {cum_return:.2%}")

es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = model_20d.fit(
    X, y20,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[es, ReturnEvalCallback()],
    verbose=1
)

# Save
model_20d.save(RESULT_DIR / "transformer_regressor_20d.keras")
pd.DataFrame(history.history).to_csv(RESULT_DIR / "training_log_reg.csv", index=False)

preds = model_20d.predict(X).flatten()
pd.DataFrame({
    "Date": all_features.index[SEQ_LEN:SEQ_LEN+len(preds)],
    "True_20d": y20,
    "Pred_20d": preds
}).to_csv(RESULT_DIR / "predictions_reg.csv", index=False)


In [ ]:
import pandas as pd
import numpy as np
import talib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import os, time
from sklearn.preprocessing import StandardScaler

# ─────────────────────────────
# 1. 경로 설정
# ─────────────────────────────
print("🔧 Stage 1: Setting paths...")
SUMMARY_FILE = Path(r"C:\Users\seung\OneDrive\주식\Financial_Data_real\Summary\2025-08-30_NVDA_nvidia_stock_summary.xlsx")
MACRO_DIR = Path(r"C:\Users\seung\OneDrive\주식\Macro Data")
RESULT_DIR = Path(r"C:\Users\seung\OneDrive\주식\Transformer Result")
os.makedirs(RESULT_DIR, exist_ok=True)

# ─────────────────────────────
# 2. 회사 데이터 불러오기
# ─────────────────────────────
print("📂 Stage 2: Loading company data...")
# Price_History 불러오기
price_df = pd.read_excel(SUMMARY_FILE, sheet_name="Price_History")
price_df = price_df.rename(columns={"날짜": "Date"})
price_df["Date"] = pd.to_datetime(price_df["Date"], errors="coerce")
price_df = price_df.set_index("Date").sort_index()

# Quarterly / Inputs 불러오기
quarterly_df = pd.read_excel(SUMMARY_FILE, sheet_name="Quarterly", parse_dates=["Date"])
inputs_df = pd.read_excel(SUMMARY_FILE, sheet_name="Inputs", parse_dates=["Date"])

quarterly_df = quarterly_df.set_index("Date").sort_index()
inputs_df = inputs_df.set_index("Date").sort_index()

# price_df index에 맞춰 reindex 후 ffill
quarterly_daily = quarterly_df.reindex(price_df.index, method="ffill")
inputs_daily = inputs_df.reindex(price_df.index, method="ffill")

# Join
company_df = price_df.join(quarterly_daily, how="left", rsuffix="_q")
company_df = company_df.join(inputs_daily, how="left", rsuffix="_i")
company_df = company_df.ffill()

print(f"   - company_df shape: {company_df.shape}")

# ─────────────────────────────
# 3. 기술적 지표 추가
# ─────────────────────────────
print("📊 Stage 3: Adding technical indicators...")
close, high, low, volume = company_df["종가"].values, company_df["고가"].values, company_df["저가"].values, company_df["거래량"].values
company_df["RSI_14"] = talib.RSI(close, timeperiod=14)
company_df["MACD"], company_df["MACD_SIG"], company_df["MACD_HIST"] = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
company_df["BB_UPPER"], company_df["BB_MID"], company_df["BB_LOWER"] = talib.BBANDS(close, timeperiod=20)
company_df["MA20"] = talib.SMA(close, timeperiod=20)
company_df["MA50"] = talib.SMA(close, timeperiod=50)
company_df["MA200"] = talib.SMA(close, timeperiod=200)
company_df["OBV"] = talib.OBV(close, volume)

# ─────────────────────────────
# 4. 매크로 데이터 불러오기
# ─────────────────────────────
print("🌍 Stage 4: Loading macro data...")
macro_files = {
    "CPI": "1970.01.01_2025.08.01 CPI.csv",
    "FedFunds": "1970.01.01_2025.08.01 FedFundsRate.csv",
    "Unemployment": "1970.01.01_2025.08.01 Unemployment.csv",
    "YieldCurve": "1970.01.01_2025.08.01 YieldCurve.csv",
    "WTI": "1986.01.01_2025.08.01 WTI.csv",
    "VIX": "1990.01.02_2025.09.18 VIX.csv",
    "HY_Spread": "1996.12.31_2025.09.18 HY_Spread.csv"
}
macro_dfs = []
for name, file in macro_files.items():
    print(f"   - Loading {name}...")
    df = pd.read_csv(MACRO_DIR / file, parse_dates=["Date"])
    df = df.set_index("Date").sort_index().resample("D").ffill()
    df = df.rename(columns={"Value": name})
    macro_dfs.append(df)
macro_df = pd.concat(macro_dfs, axis=1)
print(f"   - macro_df shape: {macro_df.shape}")

# ─────────────────────────────
# 5. CSV 저장
# ─────────────────────────────
print("💾 Saving intermediate DataFrames...")
company_df.to_csv(RESULT_DIR / "company_df.csv", encoding="utf-8-sig")
macro_df.to_csv(RESULT_DIR / "macro_df.csv", encoding="utf-8-sig")

print("🎉 Saved company_df.csv and macro_df.csv to:", RESULT_DIR)
